# Reproducing Symanski, Bladon et al. (2022) from dandiset 001539

**Paper.** Symanski CA, Bladon JH, Kullberg ET, Miller P, Jadhav SP (2022). Rhythmic coordination
and ensemble dynamics in the hippocampal-prefrontal network during odor-place associative memory
and decision making. *eLife* 11:e79545.
[doi:10.7554/eLife.79545](https://doi.org/10.7554/eLife.79545). The claims file
(`claims/001539/10.7554-elife.79545.yaml`) was written from the version of record (PMC9799972.1),
with its figure legends, Methods and supplementary file 1, and the paper's code
(JadhavLab/Jadhav-Lab-Codes-JHB at commit 88d5f8d2).

**Dandiset.** [DANDI:001539](https://dandiarchive.org/dandiset/001539/0.250815.1203), version
0.250815.1203. The paper (2022) predates the dandiset (2025) and cites no version of it. The
dandiset has two published versions, 0.250804.1538 and 0.250815.1203. They hold the same 38 assets
(same asset identifiers, paths and sizes) and differ only in their metadata (contributor roles and
the publication date), so the most recent one is pinned, as the protocol asks when the paper cites
none.

**What the dandiset holds.** One NWB file per recording day of 8 rats (38 days, the paper's 38
sessions): the task epochs of the day (no sleep epochs), a trial table (odor port entry and exit,
rewarded or not, reward well entry and exit), position (x, y, speed), sorted units and LFP at
1.5 kHz from CA1, PFC and OB tetrodes. It does not hold the odor identity of a trial, the chosen
well, spike waveforms, the nasal thermocouple, or the air-only sessions.

| Claim | Finding | Tested or blocked |
|---|---|---|
| fig1b-performance | Performance 77.0% over rats | tested |
| fig1c-sampling-duration | Sampling 0.82 s, same on correct and incorrect trials | tested |
| fig1d-turn-direction | Turn at the port matches the chosen arm | tested |
| fig1s1e-sniff-rate | Sniff rate rises during sampling | blocked: data missing |
| fig1s1f-air-performance | Performance lower on air-cued sessions | blocked: data missing |
| fig2d-beta-power-increase | Beta power rises during sampling | tested |
| fig2d-rr-power-no-increase | RR power does not rise; CA1 falls | tested |
| fig2e-beta-coherence-increase | Beta coherence rises, all pairs | tested |
| fig2e-rr-coherence-increase | RR coherence rises, all pairs | tested |
| fig2e-beta-coherence-odor-vs-reward | Beta coherence higher at the port than at reward | tested |
| fig2e-rr-coherence-odor-vs-reward | RR coherence higher at the port for OB pairs only | blocked: methods underspecified |
| fig2d-power-odor-vs-reward | Beta power differs odor vs reward; RR only in OB | tested |
| fig2de-correct-incorrect-no-difference | No power or coherence difference, correct vs incorrect | blocked: methods underspecified |
| fig2g-air-beta-coherence | Beta coherence lower on air trials | blocked: data missing |
| fig2f-swr-odor-vs-reward | Ripples rare at the port, frequent at reward | tested |
| fig2s1f-rr-beta-coupling | RR-beta phase-amplitude coupling | tested |
| fig2s2d-phase-offsets | CA1 leads PFC (beta, RR) and OB (RR) | tested |
| fig2s1e-run-theta-shift | Peak frequency shifts to theta during run | blocked: methods underspecified |
| unit-counts | Unit, cell-type and selection counts | tested |
| fig3-choice-selective-fraction | About a third of cells choice selective | tested |
| fig3ij-si-anticorrelated | Correct and incorrect SI anti-correlated | tested |
| fig3s1e-nonselective-si | Non-selective cells (Results against legend) | blocked: methods underspecified |
| fig3s1c-odor-vs-reward-rate | Rates differ, odor vs reward | tested |
| fig4c-interneuron-si | CA1 interneuron SI anti-correlated | tested |
| fig4d-ccg-lags | PFC interneurons lead in CA1-PFC CCGs | tested |
| fig5c-local-locking | Local RR and beta locking | tested |
| fig5d-cross-region-beta | CA1 interneurons lock to PFC beta | tested |
| fig5gh-phase-preference | Consistent preferred phases | tested |
| fig5-rr-more-than-beta | More and deeper RR locking than beta | tested |
| fig5s1de-beta-rr-overlap | Beta and RR locking overlap at chance | tested |
| fig5s1fh-selective-and-locked | Selective and locked at chance | tested |
| fig5e-beta-mvl-correct-incorrect | Beta locking weaker on incorrect trials | tested |
| fig5f-rr-mvl-no-difference | RR locking same on correct and incorrect | tested |
| fig5s2-ca1-int-pfc-beta-accuracy | CA1 interneuron to PFC beta locking and accuracy | tested |
| fig6d-discrimination-time | Ensemble discrimination times | tested |
| fig6c-even-odd-control | Even-odd control never significant | tested |
| fig6e-glm-choice-prediction | GLM predicts choice before exit | blocked: methods underspecified |
| fig6e-glm-with-interneurons | Interneurons make CA1 prediction earlier | blocked: methods underspecified |
| fig7-spatial-field-counts | Cells with place fields | tested |
| fig7b-odor-active-more-fields | Odor-active cells more often have fields | tested |
| fig7s1b-odor-active-higher-rate | Odor-active cells fire more in run | tested |
| fig7c-field-properties-no-difference | Field properties do not differ | tested |
| fig7s1e-trajectory-selective-odor-active | Trajectory-selective cells more often odor active | tested |
| fig7d-trajectory-choice-unrelated | Trajectory and choice preference unrelated | tested |
| fig7e-odor-decoder-first-quintile | Odor-trained decoder works in the first fifth only | tested |
| fig7f-spatial-decoder-all-quintiles | Run-trained decoder works everywhere | tested |
| fig7s1f-ensemble-rate-near-port | CA1 selective ensemble rate highest near the port | tested |

In [ ]:
import os
import time
import warnings

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import opa

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)
FULL = ["CS31", "CS33", "CS34", "CS35", "CS44"]  # the full T-maze animals (see the data check)
SHORT_STEM = ["CS39"]  # the 40 cm stem animal
ADJ = ["CS41", "CS42"]  # wells beside the odor port

In [ ]:
started = time.time()
assets = opa.list_assets()
S = opa.load_sessions(assets)
print(f"loaded {len(S)} files of dandiset {opa.DANDISET} version {opa.VERSION} in {time.time() - started:.0f} s")

In [ ]:
# Stream the LFP windows of each session and reduce them to the measures (opa.lfp_measures); the
# raw windows are not kept.
from concurrent.futures import ProcessPoolExecutor


def _measures(s):
    import opa as _opa

    return _opa.lfp_measures(s, _opa.lfp_windows(s, seed=int(s.subject[2:]) * 100 + int(s.session)))


t0 = time.time()
with ProcessPoolExecutor(4) as ex:
    M = dict(zip([s.name for s in S], ex.map(_measures, S)))
print(f"LFP measures of {len(M)} sessions in {time.time() - t0:.0f} s")

## Data check

Counts by subject, session and condition, the units of every variable used, and each check of
the metadata against the values.

In [ ]:
info = pd.DataFrame(
    [
        dict(
            name=s.name,
            subject=s.subject,
            session=s.session,
            epochs=len(s.epochs),
            epoch_types="/".join(sorted(set(s.epoch_types))),
            epoch_min=round(float((s.epochs[:, 1] - s.epochs[:, 0]).sum() / 60), 1),
            trials=len(s.trials),
            rewarded=int((s.trials.rewarded == 1).sum()),
            units=len(s.units),
            ca1=int((s.units.region == "CA1").sum()),
            pfc=int((s.units.region == "PFC").sum()),
            ob=int((s.units.region == "OB").sum()),
            row_ca1=int((s.units.row_region == "CA1").sum()),
            row_pfc=int((s.units.row_region == "PFC").sum()),
            row_ob=int((s.units.row_region == "OB").sum()),
            lfp_cols=s.lfp_shape[1],
            stem_cm=round(s.stem),
            wells_dx_cm=round(s.well_dx),
        )
        for s in S
    ]
)
print(info.to_string(index=False))
print("\nsubjects:", info.subject.nunique(), "; sessions:", len(info), "; per subject:", info.groupby("subject").size().to_dict())
print("epoch types:", sorted(set(sum([s.epoch_types for s in S], []))))
print("trials:", info.trials.sum(), "; rewarded:", info.rewarded.sum())
print("units, region from the tetrode label:", {r: int(info[r.lower()].sum()) for r in ("CA1", "PFC", "OB")})
print("units, region from the electrode-table row:", {r: int(info["row_" + r.lower()].sum()) for r in ("CA1", "PFC", "OB")})
print("Paper: 1,309 CA1 and 717 PFC units; 934 and 508 active (at least 100 spikes in run epochs); OB 'only LFP'.")

The units table's `electrodes` column is meant to hold a row index of the electrode table. Read
that way, 131 units sit on OB tetrodes, where the paper recorded no units (Methods: "Only LFP
activity was recorded from the olfactory bulb"), and the regions give 540 CA1 and 752 PFC units.
The values match the electrode table's tetrode labels (`tetrodeN`): every value is a tetrode
number present in the file (checked below), and read as the tetrode number they put no unit in
OB and give 915 CA1 and 508 PFC units. The paper's active units are 934 and 508. The second
reading is used throughout, and the first is recorded as a finding. The dandiset appears to hold
the active units, not the 1,309 and 717 of the paper (the spike counts are printed below; a few
units fall under 100 spikes within the task epochs, see unit-counts).

In [ ]:
ok_label = all(set(s.units.tetrode) <= set(s.channels.tetrode) for s in S)
print("every unit electrode value is a tetrode label number of its file:", ok_label)
print("files where every value is also a valid electrode-table row:", sum(int(s.units.tetrode.max()) < len(s.channels) for s in S), "of", len(S))
print("group name and label number differ (e.g. group Tetrode5 labelled tetrode6) in", sum((s.channels.group.str.replace("Tetrode", "").astype(int) != s.channels.tetrode).any() for s in S), "files")
nsp = np.concatenate([[opa.in_epochs(u, s.epochs) for u in s.units.spikes] for s in S])
print("spikes per unit in the task epochs: min", nsp.min(), "median", int(np.median(nsp)), "max", nsp.max(), "; units under 100:", int((nsp < 100).sum()))
print("spike times sorted in every unit:", all(s.units.sorted.all() for s in S))
print("LFP unit attribute:", sorted(set(s.lfp_unit for s in S)), "; acquisition ElectricalSeries equals the LFP on a 2000-sample stretch in", sum(s.acq_equals_lfp for s in S), "of", len(S), "files")
fs = np.array([1 / np.median(np.diff(s.lfp_ts[:20000])) for s in S])
print("LFP sampling rate from timestamps: %.2f to %.2f Hz (Methods: 1.5 kHz)" % (fs.min(), fs.max()))
gaps = [int((np.diff(s.lfp_ts) > 1).sum()) for s in S]
print("LFP gaps over 1 s (between epochs) per file:", gaps)
print("LFP chunking:", set(str(s.lfp_chunks) for s in S), "dtype", set(str(s.lfp_dtype) for s in S))
refs = pd.concat([s.channels.assign(name=s.name) for s in S])
print("LFP channels per region and reference:")
print(refs.groupby(["region", "ref"]).size().to_string())
print("position unit attribute:", sorted(set(s.pos_unit for s in S)))
pos_dt = np.array([np.median(np.diff(s.pos.t)) for s in S])
print("position frame interval: %.4f to %.4f s (Methods: 30 frames/s)" % (pos_dt.min(), pos_dt.max()))
v99 = {s.name: round(float(np.percentile(s.pos.v, 99)), 1) for s in S}
print("speed, 99th percentile per session (cm/s):", v99)
d = np.concatenate([s.trials.dur.values for s in S])
print("odor sampling duration: min %.3f, median %.3f, max %.3f s; trials under 0.5 s: %d" % (d.min(), np.median(d), d.max(), (d < 0.5).sum()))
late = {s.name: int((s.trials.stop > s.lfp_ts[-1]).sum()) for s in S if (s.trials.stop > s.lfp_ts[-1]).any()}
print("trials ending after the last LFP sample:", late)

Clock and geometry checks. The position at odor port entry should sit at one place (the odor
port), and at reward well exit at one of two places (the wells); rewarded trials give the well
side. The chosen side of every trial is read from the position after odor port exit (closest well
within 6 s, `opa.choice_side`), since the trial table has no chosen well and no odor; on rewarded
trials it is compared with the well at reward exit.

In [ ]:
rows = []
for s in S:
    tr = s.trials
    g = [s.geom[int(e)] for e in tr.epoch]
    px = np.interp(tr.start, s.pos.t, s.pos.x) - [gg["port"][0] for gg in g]
    py = np.interp(tr.start, s.pos.t, s.pos.y) - [gg["port"][1] for gg in g]
    rw = (tr.rewarded == 1).values
    rows.append(
        dict(
            name=s.name,
            port_spread_cm=round(float(np.median(np.hypot(px, py))), 1),
            side_rule_agrees=round(float((tr.side_pos == s.reward_side)[rw].mean()), 3),
            no_side=int((tr.side == "").sum()),
            odor_L_frac=round(float((tr.odor == "L").mean()), 2),
            epochs_geom=len(s.geom),
        )
    )
geo = pd.DataFrame(rows)
print(geo.to_string(index=False))
print("rewarded trials where the position rule agrees with the reward well: %.4f" % np.average(geo.side_rule_agrees, weights=[(s.trials.rewarded == 1).sum() for s in S]))
for s in S:
    if len(s.geom) > 1:
        ports = [np.round(g["port"]).tolist() for g in s.geom.values()]
        if np.ptp(np.array(ports), axis=0).max() > 5:
            print(s.name, "odor port position differs between epochs:", ports)
print("CS39_02 position range (cm):", [round(float(np.ptp(S[[x.name for x in S].index("CS39_02")].pos[c]))) for c in ("x", "y")],
      "; other CS39 days:", [[round(float(np.ptp(x.pos[c]))) for c in ("x", "y")] for x in S if x.subject == "CS39" and x.session != "02"])

Maze geometry: stems of 75 to 83 cm in CS31, CS33, CS34, CS35 and CS44 (CS44 65 to 68 cm, measured
from the port to the wells as tracked), 24 to 28 cm in CS39 and under 10 cm, with wells beside the
port, in CS41 and CS42. This fits the Methods: five animals on the full T-maze (81 cm stem), one
with a shortened stem (40 cm) and two with wells beside the odor port. The full-maze days plus the
CS39 days are 26, the Results' "26 sessions from 6 rats".

The clock check for spikes uses the trial table: task responsiveness (below) is computed with the
true odor onsets and with onsets shifted by 10 s. For the LFP, the odor-period rise in OB beta
power (fig2d) and spike phase locking (fig5) are task- and spike-locked contrasts; both appear
with the true times. These are consistent with a shared clock.

## fig1b-performance

> During recordings the rats performed the odor-place task at a high level, 77.0% correct (mean and
> s.e.m. over the 8 rats, 1.3%).

Performance is the fraction of rewarded trials per recording day (criterion). The criterion's
readings with and without trials under 0.5 s coincide here: the files hold no such trial (data
check).

In [ ]:
perf = pd.DataFrame([dict(subject=s.subject, session=s.session, perf=100 * (s.trials.rewarded == 1).mean(), n=len(s.trials)) for s in S])
per_rat = perf.groupby("subject").perf.mean()
print(per_rat.round(1).to_string())
print("(a) mean over rats of per-rat mean: %.2f%% (s.e.m. %.2f, n = %d rats)" % (per_rat.mean(), per_rat.sem(), len(per_rat)))
print("(b) mean over sessions: %.2f%% (s.e.m. %.2f, n = %d sessions)" % (perf.perf.mean(), perf.perf.sem(), len(perf)))
print("pooled over trials: %.1f%%" % (100 * np.mean(np.concatenate([s.trials.rewarded.values for s in S]))))
print("per-session range: %.1f to %.1f%%" % (perf.perf.min(), perf.perf.max()))
print("expected 77.0 +/- 1.3")

**Result: reproduced.** Reading (a), the mean over the 8 rats of each rat's mean session
performance, is 75.7% (s.e.m. 1.0), at the lower edge of 77.0 +/- 1.3 (printed to two decimals
above); reading (b), the mean over the 38 sessions, is 76.1% (s.e.m. 1.1), inside it.

## fig1c-sampling-duration

> Odor sampling lasted 0.82 s on average (s.e.m. across sessions 0.02 s; s.d. across trials 0.25 s)
> and did not differ between correct and incorrect trials.

In [ ]:
dur = pd.DataFrame([dict(subject=s.subject, name=s.name, mean=s.trials.dur.mean(), corr=s.trials.dur[s.trials.rewarded == 1].mean(), inc=s.trials.dur[s.trials.rewarded == 0].mean()) for s in S])
alld = np.concatenate([s.trials.dur.values for s in S])
print("mean over sessions %.3f s, s.e.m. %.3f (n = %d); s.d. across trials %.3f (n = %d trials)" % (dur["mean"].mean(), dur["mean"].sem(), len(dur), alld.std(ddof=1), len(alld)))
print("s.d. across trials within sessions, mean over sessions: %.3f" % np.mean([s.trials.dur.std() for s in S]))
print("s.d. across trials, trials of 0.5 to 2 s only: %.3f" % alld[alld <= 2].std(ddof=1))
w = stats.wilcoxon(dur["corr"], dur.inc)
print("correct %.3f vs incorrect %.3f s (means over sessions); median paired difference %.3f s; signed-rank p = %.3g (n = %d)" % (dur["corr"].mean(), dur.inc.mean(), np.median(dur["corr"] - dur.inc), w.pvalue, len(dur)))
print("variant, rank-sum on the per-session means: p = %.3g" % stats.ranksums(dur["corr"], dur.inc).pvalue)
print(dur.groupby("subject")[["mean", "corr", "inc"]].mean().round(3).to_string())
print("expected 0.82 +/- 0.02 s and 0.25 +/- 0.025 s; absence judged on the signed-rank test")

**Result: reproduced.** The mean sampling duration over the 38 sessions is 0.819 s (s.e.m.
0.017) and the s.d. across the 4,182 trials 0.245 s, both within tolerance. Correct and incorrect
trials do not differ (signed-rank p = 0.72 over 38 sessions, the paper 0.71; median paired
difference 0.016 s; the rank-sum variant gives p = 0.55).

## fig1d-turn-direction

> On most trials the rat turned away from the odor port toward the side of the reward arm it then
> chose, on correct and incorrect trials alike, so the choice is made during odor sampling.

Turn direction as the criterion gives it (F16/Behavior/cs_turnFromNPDirection.m). The chosen arm
is the well reached after exit (data check). Sessions on the truncated mazes are excluded
(legend). Whether CS39 (40 cm stem, wells at the ends of the T arms) counts as truncated is not
said: the legend excludes "the truncated task without spatial delay period", and CS39's maze has a
shorter delay. Both sets are shown.

In [ ]:
rows = []
for s in S:
    turn = opa.turn_direction(s)
    tr = s.trials
    for i in range(len(tr)):
        if turn[i] == "" or tr.side.values[i] == "":
            continue
        rows.append(dict(subject=s.subject, name=s.name, correct=tr.rewarded.values[i] == 1, same=turn[i] == tr.side.values[i]))
turns = pd.DataFrame(rows)
for label, subs in (("full maze (5 rats)", FULL), ("full maze and CS39 (6 rats)", FULL + SHORT_STEM)):
    d = turns[turns.subject.isin(subs)]
    print(label, "; trials with a turn:", len(d))
    for c, name in ((True, "correct"), (False, "incorrect")):
        x = d[d.correct == c]
        k = int(x.same.sum())
        n = len(x)
        print("  %s: same %d, opposite %d, binomial p = %.3g" % (name, k, n - k, stats.binomtest(k, n, 0.5).pvalue))
    by = d.groupby("name").same.mean()
    print("  fraction same, (a) mean over sessions %.1f%% (s.e.m. %.2f%%), (b) pooled %.1f%%" % (100 * by.mean(), 100 * by.sem(), 100 * d.same.mean()))
    print("  per animal:", d.groupby("subject").same.mean().round(3).to_dict())
print("expected: 95.4% (+/- 0.12), correct 1624 / 32, incorrect 499 / 21 (tolerances 162, 3, 50, 2)")
print("all 8 rats, for reference: same %.1f%%" % (100 * turns.same.mean()))

**Result: partially reproduced.** The turn matches the chosen arm on most trials and both
binomial tests are significant in both sets. With CS39 included the trial counts are close to the
paper's (1,662 correct and 520 incorrect trials against 1,656 and 520), and the same-direction
counts are within tolerance (1,615 and 490), but the opposite-direction counts are not (47 and 30
against 32 +/- 3 and 21 +/- 2). The fraction same is 95.9% over sessions and 96.5% pooled, against
95.4 +/- 0.12. On the five full-maze rats alone the counts are further off (1,390 and 30; 430 and
24).

## fig1s1e-sniff-rate, fig1s1f-air-performance and fig2g-air-beta-coherence

Not tested. The files have no thermocouple channel (the electrode table lists only CA1, PFC and OB
tetrodes), and every epoch is an odor-place epoch: the air-only sessions are not in the dandiset
(data check). The 3,174 rewarded trials of the files equal the 3,174 odor-cued correct trials of
the Figure 2g legend, so the odor-cued side of fig2g is all there; the 134 air trials are not.

## LFP power and coherence (fig2d, fig2e)

The measures come from `opa.lfp_measures`, under the two readings of the claims file's opening
comment. Reading A: 1 s spectral windows stepped by 20 ms, tapers [1 1] for power and [2 3] for
coherence, every tetrode of a region with cells that day (all OB tetrodes) averaged, all
cross-region pairs averaged, each z-scored per frequency to the epoch mean and s.d.; the odor window
is the whole sampling period and the pre-odor window has the same length and ends at odor onset.
Reading B: tapers [3 5] on the last 500 ms of sampling and the 500 ms before odor onset, trials of
0.5 to 2 s, the tetrode with most units per region, the LFP z-scored over the recording.

Two details are not in the claims file. The trials: neither the Methods nor the criterion restricts
the pre-odor comparison to correct trials, so all trials are used, and correct trials only are
shown as a variant. The epoch mean and s.d. for z-scoring are estimated from 60 windows of 2 s at
random times in each day's task epochs (an assumption made to keep the run within its time limit;
a within-session difference keeps its sign under any such scaling, so only its size changes).

In [ ]:
def sess_table(field_a, field_b, trial_sel=None, src="trials"):
    """Per session, the mean over selected trials of field_a minus field_b (columns of the
    per-trial measures). trial_sel(s) gives a boolean mask over the session's trials."""
    out = []
    for s in S:
        m = M[s.name][src]
        tr = s.trials
        if field_a not in m or field_b not in m:
            continue
        sel = np.ones(len(m), bool) if trial_sel is None else trial_sel(s)[m.trial.values]
        d = (m[field_a] - m[field_b])[sel].dropna()
        out.append(dict(name=s.name, subject=s.subject, diff=d.mean(), n=len(d)))
    return pd.DataFrame(out)


def B_ok(s):
    return ((s.trials.dur >= 0.5) & (s.trials.dur <= 2.0)).values


def correct(s):
    return (s.trials.rewarded == 1).values


def test_rows(measure, bands, regions, a, b, sel_a=None, sel_b=None, label=""):
    rows = []
    for bn in bands:
        for r in regions:
            for rd, fa, fb, sel in (("A", f"A_{a}_{measure}_{r}_{bn}", f"A_{b}_{measure}_{r}_{bn}", sel_a), ("B", f"B_{a if a != 'odor' else 'last'}_{measure}_{r}_{bn}", f"B_{b}_{measure}_{r}_{bn}", sel_b)):
                t = sess_table(fa, fb, sel)
                t = t[np.isfinite(t["diff"])]
                p = stats.wilcoxon(t["diff"]).pvalue
                rows.append(dict(reading=rd, band=bn, what=r, sessions=len(t), median_diff=round(float(t["diff"].median()), 4), mean_diff=round(float(t["diff"].mean()), 4), n_pos=int((t["diff"] > 0).sum()), p=float(p)))
    return pd.DataFrame(rows)


REG = ["CA1", "PFC", "OB"]
PAIRS = ["CA1-PFC", "CA1-OB", "PFC-OB"]
pow_pre = test_rows("pow", ["beta", "rr"], REG, "odor", "pre", None, B_ok)
pow_pre_corr = test_rows("pow", ["beta", "rr"], REG, "odor", "pre", correct, lambda s: B_ok(s) & correct(s))
coh_pre = test_rows("coh", ["beta", "rr"], PAIRS, "odor", "pre", None, B_ok)
coh_pre_corr = test_rows("coh", ["beta", "rr"], PAIRS, "odor", "pre", correct, lambda s: B_ok(s) & correct(s))

### fig2d-beta-power-increase

> Beta power rose from the time-matched pre-odor period to the odor sampling period in CA1, PFC and OB.

In [ ]:
print("odor minus pre-odor power, per session; signed-rank across sessions (all trials; reading B trials of 0.5 to 2 s)")
print(pow_pre[pow_pre.band == "beta"].to_string(index=False))
print("\nvariant, correct trials only:")
print(pow_pre_corr[pow_pre_corr.band == "beta"].to_string(index=False))
print("\nper animal, reading A, mean of session differences:")
print(pd.concat({r: sess_table(f"A_odor_pow_{r}_beta", f"A_pre_pow_{r}_beta").groupby("subject")["diff"].mean() for r in REG}, axis=1).round(3).to_string())
print("paper: CA1 p = 2.84e-3, PFC p = 1.80e-4, OB p = 8.3e-6 (n = 38)")

**Result: reproduced.** Beta power rises from the pre-odor window to odor sampling in all three regions under both readings (reading A: CA1 p = 9.0e-7, PFC p = 1.3e-8, OB p = 7.3e-12; reading B: p = 1.2e-4, 3.3e-9 and 3.6e-11; 31 to 38 of 38 sessions positive). The outcome rests on the epoch statistics for z-scoring being estimated from 120 s of each day.

### fig2d-rr-power-no-increase

> RR power did not rise after odor onset; it fell slightly in CA1 and did not change in PFC or OB.

In [ ]:
print(pow_pre[pow_pre.band == "rr"].to_string(index=False))
print("\nvariant, correct trials only:")
print(pow_pre_corr[pow_pre_corr.band == "rr"].to_string(index=False))
print("paper: CA1 decrease p = 3.94e-3, PFC p = 0.551, OB p = 0.949")

**Result: not reproduced.** Neither reading finds the CA1 decrease (reading A p = 0.099, reading B p = 0.21, both with the median below zero). Reading A finds RR increases in OB (p = 9.4e-8) and PFC (p = 0.049); reading B finds no significant change in any region (OB p = 0.057). The criterion (no increase, a CA1 decrease, no PFC or OB change) is met under neither reading.

### fig2e-beta-coherence-increase

> Beta coherence between all three region pairs rose from the pre-odor period to odor sampling.

In [ ]:
print(coh_pre[coh_pre.band == "beta"].to_string(index=False))
print("\nvariant, correct trials only:")
print(coh_pre_corr[coh_pre_corr.band == "beta"].to_string(index=False))
print("\nper animal, reading A:")
print(pd.concat({r: sess_table(f"A_odor_coh_{r}_beta", f"A_pre_coh_{r}_beta").groupby("subject")["diff"].mean() for r in PAIRS}, axis=1).round(3).to_string())
print("paper: CA1-PFC p = 2.8e-3, CA1-OB p = 5.1e-5, PFC-OB p = 3.7e-5")

**Result: reproduced.** Beta coherence rises for all three pairs under both readings (reading A: CA1-PFC p = 3.7e-7, CA1-OB p = 5.1e-11, PFC-OB p = 1.4e-10; reading B: p = 2.6e-5, 1.2e-7, 1.1e-8).

### fig2e-rr-coherence-increase

> RR coherence also rose from the pre-odor period to odor sampling for all three region pairs.

In [ ]:
print(coh_pre[coh_pre.band == "rr"].to_string(index=False))
print("\nvariant, correct trials only:")
print(coh_pre_corr[coh_pre_corr.band == "rr"].to_string(index=False))
print("paper: CA1-OB p = 6.4e-4, CA1-PFC p = 3.9e-3, PFC-OB p = 2.4e-4")

**Result: partially reproduced.** RR coherence rises for CA1-OB (reading A p = 3.9e-5, B p = 3.0e-4) and PFC-OB (p = 3.6e-8 and 5.8e-5) but not for CA1-PFC (p = 0.11 and 0.10), under both readings; two of the three pairs hold.

### Odor sampling against the reward well (fig2e middle, fig2d middle)

Last 500 ms of odor sampling against the last 500 ms before reward well exit, rewarded trials,
per session odor minus reward, signed-rank across sessions (criterion). Reading B is the code's
pipeline (tapers [3 5], one tetrode per region); reading A's settings are the 1 s spectral windows
whose centres fall in each 500 ms window. Reward well exit is the trial table's
`reward_end_time` ("Time of reward well exit"). The code's alternative exit time (the next DIO
event after the pump turns off) and the windows centred on exit are not available here: the files
hold no DIO events. Sessions without the reward measure for every rewarded trial use the trials
that have it.

In [ ]:
def odor_vs_reward(measure, bands, regions):
    rows = []
    for bn in bands:
        for r in regions:
            for rd in ("A", "B"):
                out = []
                for s in S:
                    m = M[s.name]["trials"].set_index("trial")
                    rw = M[s.name]["reward"].set_index("trial")
                    fo = f"{rd}_last_{measure}_{r}_{bn}"
                    fr = f"{rd}_rew_{measure}_{r}_{bn}"
                    if fo not in m or fr not in rw:
                        continue
                    idx = rw.index[(s.trials.rewarded.values[rw.index] == 1)]
                    if rd == "B":
                        idx = idx[B_ok(s)[idx]]
                    d = (m.loc[idx, fo] - rw.loc[idx, fr]).dropna()
                    out.append(d.mean())
                out = np.array(out)
                out = out[np.isfinite(out)]
                rows.append(dict(reading=rd, band=bn, what=r, sessions=len(out), median_diff=round(float(np.median(out)), 4), n_pos=int((out > 0).sum()), p=float(stats.wilcoxon(out).pvalue)))
    return pd.DataFrame(rows)


coh_rew = odor_vs_reward("coh", ["beta", "rr"], PAIRS)
pow_rew = odor_vs_reward("pow", ["beta", "rr"], REG)

### fig2e-beta-coherence-odor-vs-reward

> Beta coherence in the last 500 ms of odor sampling was higher than in the last 500 ms before
> reward well exit, for all three region pairs, so it is not a sign of movement preparation.

In [ ]:
print("odor minus reward, per session")
print(coh_rew[coh_rew.band == "beta"].to_string(index=False))
print("paper: CA1-PFC p = 0.010, CA1-OB p = 0.0012, PFC-OB p = 0.0024")

**Result: reproduced.** Beta coherence is higher in the last 500 ms of sampling than in the last 500 ms before reward well exit for all three pairs under both settings (reading A p = 3.1e-10, 1.5e-11, 5.5e-9; reading B p = 2.2e-9, 3.9e-9, 3.8e-6; 33 to 37 of 38 sessions higher at the odor port).

### fig2e-rr-coherence-odor-vs-reward

> For RR, coherence was higher during odor sampling than at the reward well for CA1-OB and PFC-OB
> but not for CA1-PFC.

In [ ]:
print(coh_rew[coh_rew.band == "rr"].to_string(index=False))
print("paper: CA1-PFC p = 0.569, CA1-OB p = 0.0044, PFC-OB p = 0.00064")

**Result: blocked (methods underspecified).** The two settings give different outcomes. Under reading A all three parts hold: CA1-OB higher at the odor port (p = 0.010), PFC-OB higher (p = 1.8e-10), CA1-PFC not significant (p = 0.055). Under reading B only CA1-OB holds (p = 0.0012); PFC-OB is not significant (p = 0.074) and CA1-PFC is (p = 0.027). The criterion blocks the claim when the settings differ.

### fig2d-power-odor-vs-reward

> Beta power in all three regions differed between the odor sampling and reward periods, while RR
> power differed only in OB.

In [ ]:
print(pow_rew.to_string(index=False))
print("paper: beta CA1 p = 0.0012, PFC p = 5.3e-4, OB p = 4.4e-4; RR CA1 p = 0.11, PFC p = 0.062, OB p = 4.4e-4")

**Result: partially reproduced.** Beta power differs between the odor and reward windows in all three regions under both settings (p from 7.3e-12 to 1.4e-6), and RR power differs in OB (p = 7.3e-12 and 1.0e-6). RR power also differs in PFC under both (p = 1.0e-10 and 0.014), and in CA1 under reading A (p = 1.0e-5; reading B p = 0.21). Four of the six parts hold under reading A and five under reading B.

### fig2de-correct-incorrect-no-difference

> Neither power nor coherence, in beta or RR, differed between correct and incorrect trials.

Correct trials subsampled at random to the number of incorrect trials in each session (seeded);
the odor window of each reading; per session correct minus incorrect. The two tests the paper names
are both run: signed-rank of the differences across sessions, and rank-sum of the per-session
correct values against the per-session incorrect values.

In [ ]:
rng = np.random.default_rng(2024)
rows = []
for rd, win in (("A", "odor"), ("B", "last")):
    for kind, names in (("pow", REG), ("coh", PAIRS)):
        for bn in ("beta", "rr"):
            for r in names:
                f = f"{rd}_{win}_{kind}_{r}_{bn}"
                c_vals, i_vals = [], []
                for s in S:
                    m = M[s.name]["trials"]
                    if f not in m:
                        continue
                    ok = np.ones(len(s.trials), bool) if rd == "A" else B_ok(s)
                    v = m.set_index("trial")[f]
                    ci = np.nonzero(ok & correct(s))[0]
                    ii = np.nonzero(ok & ~correct(s))[0]
                    ci = [x for x in ci if np.isfinite(v.get(x, np.nan))]
                    ii = [x for x in ii if np.isfinite(v.get(x, np.nan))]
                    if len(ii) < 1 or len(ci) < len(ii):
                        continue
                    pick = rng.choice(ci, len(ii), replace=False)
                    c_vals.append(v.loc[pick].mean())
                    i_vals.append(v.loc[ii].mean())
                c_vals, i_vals = np.array(c_vals), np.array(i_vals)
                rows.append(dict(reading=rd, measure=kind, band=bn, what=r, sessions=len(c_vals), median_diff=round(float(np.median(c_vals - i_vals)), 4), p_signrank=float(stats.wilcoxon(c_vals - i_vals).pvalue), p_ranksum=float(stats.ranksums(c_vals, i_vals).pvalue)))
ci_tab = pd.DataFrame(rows)
print(ci_tab.to_string(index=False))
for rd in ("A", "B"):
    t = ci_tab[ci_tab.reading == rd]
    print(f"reading {rd}: comparisons with signed-rank p < 0.05: {int((t.p_signrank < 0.05).sum())} of {len(t)}; with rank-sum p < 0.05: {int((t.p_ranksum < 0.05).sum())}")

The subsample of correct trials is random. To see how much the one reading-B result depends on it,
the subsampling is repeated with 20 other seeds (a check, not part of the criterion).

In [ ]:
def ci_once(seed, rd, win):
    rng = np.random.default_rng(seed)
    res = []
    for kind, names in (("pow", REG), ("coh", PAIRS)):
        for bn in ("beta", "rr"):
            for r in names:
                f = f"{rd}_{win}_{kind}_{r}_{bn}"
                cv, iv = [], []
                for s in S:
                    m = M[s.name]["trials"].set_index("trial")
                    if f not in m:
                        continue
                    v = m[f]
                    ok = np.ones(len(s.trials), bool) if rd == "A" else B_ok(s)
                    ci = [x for x in np.nonzero(ok & correct(s))[0] if np.isfinite(v.get(x, np.nan))]
                    ii = [x for x in np.nonzero(ok & ~correct(s))[0] if np.isfinite(v.get(x, np.nan))]
                    if len(ii) < 1 or len(ci) < len(ii):
                        continue
                    cv.append(v.loc[rng.choice(ci, len(ii), replace=False)].mean())
                    iv.append(v.loc[ii].mean())
                cv, iv = np.array(cv), np.array(iv)
                res.append((stats.wilcoxon(cv - iv).pvalue, stats.ranksums(cv, iv).pvalue))
    return np.array(res)


for rd, win in (("A", "odor"), ("B", "last")):
    rs = [ci_once(1000 + k, rd, win) for k in range(20)]
    print(f"reading {rd}: seeds with any signed-rank p < 0.05: {sum((r[:, 0] < 0.05).any() for r in rs)} of 20; with any rank-sum p < 0.05: {sum((r[:, 1] < 0.05).any() for r in rs)} of 20; smallest signed-rank p over seeds {min(r[:, 0].min() for r in rs):.3g}")

**Result: blocked (methods underspecified).** The rank-sum test the legend names finds no difference in any of the 24 comparisons (smallest p = 0.65), and the signed-rank test the Results name finds none under reading A (smallest p = 0.12) but one under reading B (PFC beta power, p = 0.023, median difference 0.22). The criterion blocks the claim when the two tests differ. The subsampling of correct trials matters: over 20 other seeds the signed-rank test finds at least one difference in 7 (reading A) and 13 (reading B), the rank-sum test in none.

## fig2f-swr-odor-vs-reward

> Sharp-wave ripples were rare during odor sampling and frequent at the reward wells.

Detection as the criterion gives it (Methods): locally referenced ("Ref ON") CA1 LFP of the CA1
tetrodes with cells, 150 to 250 Hz, Hilbert envelope above 3 s.d. of its mean for at least 15 ms
on at least one tetrode, at speed under 4 cm/s. The mean and s.d. of the envelope come from the
baseline windows of the task epochs (the files hold no sleep epochs). The band-pass is a
Butterworth of order 4, applied forward and backward (the order is not given; a standard choice).
Correct trials. Readings (a) the sampling period and the reward-well period (entry to exit);
(b) the speed-defined stops around the odor port and around reward entry, the latter extended to
at least 1 s. Per session the mean count per trial; signed-rank across sessions.

In [ ]:
rows = []
for s in S:
    r = M[s.name]["ripples"]
    c = correct(s) & np.isfinite(r.reward.values) & np.isfinite(r.odor.values)
    rows.append(dict(name=s.name, subject=s.subject, n=int(c.sum()), odor=r.odor[c].mean(), reward=r.reward[c].mean(), odor_stop=r.odor_stop[c].mean(), reward_stop=r.reward_stop[c].mean(), odor_stop_s=r.odor_stop_len[c].mean(), reward_stop_s=r.reward_stop_len[c].mean(), reward_s=(s.trials.rend - s.trials.rstart)[c].mean()))
swr = pd.DataFrame(rows)
print(swr.round(3).to_string(index=False))
for a, b, lab in (("odor", "reward", "(a) sampling period vs reward-well period"), ("odor_stop", "reward_stop", "(b) stops at the odor port vs at reward")):
    d = swr[a] - swr[b]
    print(f"{lab}: mean per trial {swr[a].mean():.3f} vs {swr[b].mean():.3f}; fewer at the odor port in {int((d < 0).sum())} of {len(d)} sessions, equal in {int((d == 0).sum())}; signed-rank p = {stats.wilcoxon(swr[a], swr[b]).pvalue:.3g}")
print("variant, rate per second, reading (a): %.3f vs %.3f per s" % ((swr.odor / [s.trials.dur[correct(s)].mean() for s in S]).mean(), (swr.reward / swr.reward_s).mean()))
sp = np.concatenate([np.interp(np.linspace(a, b, 20), s.pos.t, s.pos.v) for s in S for a, b in zip(s.trials.rstart[correct(s)], s.trials.rend[correct(s)])])
print("speed during reward-well periods: median %.1f cm/s, fraction under 4 cm/s %.2f" % (np.median(sp), (sp < 4).mean()))
print("paper: signed-rank p = 1.1e-7 (n = 38)")

**Result: reproduced.** Ripples are fewer during odor sampling than at the reward well in both readings: (a) 0.013 against 3.54 per trial, fewer at the odor port in all 38 sessions, p = 7.3e-12; (b) 0.036 against 0.80 per trial in the speed-defined stops, fewer in 37 of 38, p = 8.0e-10. The threshold rests on envelope statistics from task-epoch windows (the files hold no sleep).

## fig2s1f-rr-beta-coupling

> Beta amplitude was coupled to RR phase in CA1, PFC and OB during odor sampling.

Tort modulation index (criterion; Methods): RR (7 to 8 Hz) phase and beta (20 to 30 Hz) amplitude
of the local main tetrode, samples of the sampling periods (every third sample kept), pooled over
the trials of all sessions; each session's beta amplitude is divided by its mean so that no session
dominates the pooled amplitude (a standard choice). Null: 500 shuffles of the trial assignment of
the RR phase series within each session (Methods), a shuffled pair cut to the shorter of the two.
Readings all trials and correct trials. Per-session MIs against their own 500 shuffles are a
variant.

In [ ]:
rng = np.random.default_rng(7)


def cfc_series(region, sel):
    ph, am = [], []
    for s in S:
        c = M[s.name]["cfc"]
        keep = [i for i in np.nonzero(sel(s))[0] if region in c[i] and len(c[i][region][0]) > 0]
        a_mean = np.mean(np.concatenate([c[i][region][1] for i in keep])) if keep else 1.0
        ph.append([c[i][region][0] for i in keep])
        am.append([c[i][region][1] / a_mean for i in keep])
    return ph, am


def pooled_mi(ph, am, perm=None):
    P, A = [], []
    for k in range(len(ph)):
        idx = range(len(ph[k])) if perm is None else perm[k]
        for i, j in zip(range(len(ph[k])), idx):
            n = min(len(ph[k][i]), len(am[k][j]))
            P.append(ph[k][i][:n])
            A.append(am[k][j][:n])
    return opa.tort_mi(np.concatenate(P), np.concatenate(A))


cfc_rows = []
for sel_name, sel in (("all trials", lambda s: np.ones(len(s.trials), bool)), ("correct trials", correct)):
    for reg in REG:
        ph, am = cfc_series(reg, sel)
        mi = pooled_mi(ph, am)
        null = np.array([pooled_mi(ph, am, [rng.permutation(len(x)) for x in ph]) for _ in range(500)])
        cfc_rows.append(dict(trials=sel_name, region=reg, MI=mi, null_mean=null.mean(), null_max=null.max(), n_above=int((null >= mi).sum()), p=(1 + (null >= mi).sum()) / 501))
cfc = pd.DataFrame(cfc_rows)
print(cfc.to_string(index=False))

In [ ]:
rows = []
for reg in REG:
    ph, am = cfc_series(reg, lambda s: np.ones(len(s.trials), bool))
    for k, s in enumerate(S):
        if len(ph[k]) < 5:
            continue
        mi = pooled_mi([ph[k]], [am[k]])
        null = np.array([pooled_mi([ph[k]], [am[k]], [rng.permutation(len(ph[k]))]) for _ in range(500)])
        rows.append(dict(region=reg, name=s.name, MI=mi, p=(1 + (null >= mi).sum()) / 501))
per = pd.DataFrame(rows)
print("variant, per session (all trials): sessions with MI above 95% of its shuffles")
print(per.groupby("region").apply(lambda d: f"{int((d.p < 0.05).sum())} of {len(d)}", include_groups=False).to_string())

**Result: reproduced.** The pooled modulation index exceeds all 500 trial shuffles in CA1, PFC and OB, with all trials and with correct trials (p = 0.002, the smallest a 500-shuffle test can give). The indices are small (4.3e-5, 0.7e-5 and 1.3e-5 against shuffle maxima of 0.5e-5 to 0.7e-5). Per session, 15, 9 and 17 of 38 sessions exceed 95% of their shuffles.

## fig2s2d-phase-offsets

> During odor sampling CA1 beta led PFC beta, and CA1 RR led PFC and OB RR, though offsets varied
> widely across sessions.

From the criterion (F7/ClairePhaseOffset.m): per trial the circular mean of the instantaneous phase
difference (first region minus second; positive means the first region leads), per session the
circular mean over trials, one tetrode per region (most units), and a sign-flip null of 100,000
draws on the session offsets, taking the smaller tail. The statistic is the mean of the session
offsets (wrapped to -pi..pi). Readings (a) the whole sampling period, (b) the last 500 ms, trials of
0.5 to 2 s.

In [ ]:
rng = np.random.default_rng(11)
rows = []
for win, label in (("whole", "(a) whole sampling period"), ("last", "(b) last 500 ms")):
    for pair in PAIRS:
        for bn in ("beta", "rr"):
            f = f"{win}_{pair}_{bn}"
            sess = []
            for s in S:
                o = M[s.name]["offsets"]
                if f not in o:
                    continue
                sel = B_ok(s)[o.trial.values] if win == "last" else np.ones(len(o), bool)
                v = o[f][sel].dropna().values
                if len(v):
                    sess.append(np.angle(np.mean(np.exp(1j * v))))
            sess = np.array(sess)
            obs = sess.mean()
            flips = rng.choice([-1.0, 1.0], size=(100000, len(sess)))
            null = (flips * sess).mean(1)
            p = min((null >= obs).mean(), (null <= obs).mean())
            rows.append(dict(window=label, pair=pair, band=bn, sessions=len(sess), mean_offset_deg=round(float(np.degrees(obs)), 1), circ_mean_deg=round(float(np.degrees(np.angle(np.mean(np.exp(1j * sess))))), 1), positive=int((sess > 0).sum()), p_smaller_tail=float(p)))
off = pd.DataFrame(rows)
print(off.to_string(index=False))
print("paper: beta CA1 leads PFC p = 2e-4; RR CA1 leads PFC and OB p < 1e-5; beta OB pairs p > 0.05")

**Result: partially reproduced.** Under both windows CA1 beta leads PFC beta (mean offset 10.8 and 12.1 degrees, p = 2.0e-4 and 7e-5), CA1 RR leads PFC RR (43 and 42 degrees, p < 1e-5) and CA1 RR leads OB RR (28 and 30 degrees, p = 3e-5 and < 1e-5). PFC-OB beta has no consistent offset (p = 0.44 and 0.42), but CA1-OB beta does (20 and 23 degrees, p = 0.011 and 0.012), where the paper finds none.

## fig2s1e-run-theta-shift

> After odor port exit, as the rat ran, the dominant LFP frequency in CA1 and PFC shifted slightly
> from RR up into the theta band.

Full-maze animals (legend). Spectra of 1 s windows (tapers [3 5], the Chronux frequency grid of 0.73 Hz up to 20 Hz), the main tetrode of
CA1 and PFC; per day the mean spectrum over trials, its peak frequency between 6 and 12 Hz
(criterion), and a signed-rank test of run against odor over days. Readings (a) windows centred 0
to 0.8 s after odor onset (odor) and 0.8 to 3 s (run); (b) windows centred in the sampling period
(odor) and after exit at speed over 3 cm/s, up to 3 s after onset (run). The window step is 50 ms
(a standard choice).

In [ ]:
rows = []
fr = opa.THETA_FREQS
band = (fr >= 6) & (fr <= 12)
for s in S:
    if s.subject not in FULL:
        continue
    th = M[s.name]["theta"]
    row = dict(name=s.name)
    for rd in ("a", "b"):
        for reg in ("CA1", "PFC"):
            for part in ("odor", "run"):
                spec = [t.get(f"{rd}_{part}_{reg}") for t in th]
                spec = [x for x in spec if x is not None]
                if spec:
                    mspec = np.mean(spec, 0)
                    row[f"{rd}_{reg}_{part}"] = fr[band][np.argmax(mspec[band])]
                    row[f"{rd}_{reg}_{part}_all"] = fr[np.argmax(mspec)]
    rows.append(row)
pk = pd.DataFrame(rows)
print(pk.to_string(index=False))
for rd in ("a", "b"):
    for reg in ("CA1", "PFC"):
        o, r = pk[f"{rd}_{reg}_odor"], pk[f"{rd}_{reg}_run"]
        p = stats.wilcoxon(r, o).pvalue if (r != o).any() else 1.0
        print(f"reading {rd}, {reg}: peak odor median {o.median()} Hz, run median {r.median()} Hz; run higher in {int((r > o).sum())} of {len(r)} days, lower in {int((r < o).sum())}; signed-rank p = {p:.3g}")

**Result: blocked (methods underspecified).** The window readings give different outcomes. Under reading (a) the CA1 peak is higher during run on 7 of 19 days and never lower (p = 0.011), while the PFC peak does not move (p = 1); that is a partial result. Under reading (b) neither region shifts (CA1 p = 0.058, PFC p = 1). In PFC the peak sits at the lowest frequency searched (6.6 Hz) on every day, so the PFC spectrum peaks below the theta band in both parts.

## Single units: selection

Definitions from the claims file's opening comment, each under the readings it names:
- active: at least 100 spikes during the run epochs (the files hold only task epochs); readings
  (A) more than 100 spikes in any one epoch (Path A) and (B) at least 100 across all task epochs
  (Path B; Results);
- cell type: interneuron when the mean rate over the task epochs exceeds 7 Hz and the spike width
  is under 0.3 ms. The files hold no waveforms, so the width cannot be measured and the rate alone
  is used (as the code's F7/ParsePyrIN.m does, at 9.5 Hz); the 8.5 and 9.5 Hz thresholds of the
  code are shown as variants;
- odor period active: spikes in the sampling periods at least equal to the number of trials;
- task responsive: per odor, a signed-rank test of sampling-period rate against a pre-odor window
  of the same length; readings (a) all completed trials, (b) correct trials only.

Odor identity is not in the files. A trial's odor is read as the side of the well the rat reached
when it was rewarded, and the other side when it was not (data check). Which side is odor 1 is not
known, so a selectivity index has an arbitrary sign within a session; nothing below uses its sign
across sessions.

In [ ]:
rows = []
for s in S:
    tr = s.trials
    ep_len = (s.epochs[:, 1] - s.epochs[:, 0]).sum()
    corr_m = (tr.rewarded == 1).values
    for u, row in s.units.iterrows():
        spk = row.spikes
        per_ep = np.array([opa.count_in(spk, e[0], e[1]) for e in s.epochs])
        odor_spk = int(opa.count_in(spk, tr.start.values, tr.stop.values).sum())
        rows.append(
            dict(
                name=s.name,
                subject=s.subject,
                unit=u,
                region=row.region,
                n_total=len(spk),
                n_epochs=int(per_ep.sum()),
                active_A=bool((per_ep > 100).any()),
                active_B=bool(per_ep.sum() >= 100),
                rate=per_ep.sum() / ep_len,
                odor_active=odor_spk >= len(tr),
                p_resp_a=opa.task_responsive(spk, tr, np.ones(len(tr), bool)),
                p_resp_b=opa.task_responsive(spk, tr, corr_m),
                p_resp_a_shift=opa.task_responsive(spk, tr.assign(start=tr.start + 10, stop=tr.stop + 10), np.ones(len(tr), bool)),
            )
        )
U = pd.DataFrame(rows)
U["int7"] = U.rate > 7
U["int85"] = U.rate > 8.5
U["int95"] = U.rate > 9.5
print("units:", len(U), U.region.value_counts().to_dict())
print("clock check, task responsive (reading a) with true odor onsets: %d of %d odor-active units; with onsets shifted by 10 s: %d" % ((U.p_resp_a < 0.05)[U.odor_active].sum(), U.odor_active.sum(), (U.p_resp_a_shift < 0.05)[U.odor_active].sum()))
print("rate over task epochs (Hz): min %.2f, median %.2f, max %.1f" % (U.rate.min(), U.rate.median(), U.rate.max()))

## unit-counts

> The recordings gave 1,309 CA1 and 717 PFC units, of which 934 and 508 were active, 813 CA1 and
> 464 PFC active units were putative pyramidal cells (121 and 44 interneurons), 170 and 234
> pyramidal cells were active in the odor period, and 138 and 185 of these were task responsive.

In [ ]:
def counts(active, intcol, resp):
    out = {}
    for reg in ("CA1", "PFC"):
        d = U[U.region == reg]
        a = d[d[active]]
        pyr = a[~a[intcol]]
        it = a[a[intcol]]
        oa = pyr[pyr.odor_active]
        tr_ = oa[oa[resp] < 0.05]
        out[reg] = dict(units=len(d), n_active=len(a), pyramidal=len(pyr), interneurons=len(it), odor_active_pyr=len(oa), task_responsive_pyr=len(tr_))
    return out


expected = dict(CA1=dict(units=1309, n_active=934, pyramidal=813, interneurons=121, odor_active_pyr=170, task_responsive_pyr=138), PFC=dict(units=717, n_active=508, pyramidal=464, interneurons=44, odor_active_pyr=234, task_responsive_pyr=185))
tol = dict(CA1=dict(units=131, n_active=93, pyramidal=81, interneurons=12, odor_active_pyr=17, task_responsive_pyr=14), PFC=dict(units=72, n_active=51, pyramidal=46, interneurons=4, odor_active_pyr=23, task_responsive_pyr=19))
rows = []
for active in ("active_A", "active_B"):
    for intcol in ("int7", "int85", "int95"):
        for resp in ("p_resp_a", "p_resp_b"):
            c = counts(active, intcol, resp)
            for reg in ("CA1", "PFC"):
                r = dict(active=active[-1], rate_thr={"int7": "7", "int85": "8.5", "int95": "9.5"}[intcol], resp=resp[-1], region=reg, **c[reg])
                r["within"] = ",".join(k for k in expected[reg] if abs(c[reg][k] - expected[reg][k]) <= tol[reg][k])
                rows.append(r)
uc = pd.DataFrame(rows)
print(uc.to_string(index=False))
print("expected:", expected)
print("\nunits per subject and region (deposited):")
print(U.groupby(["subject", "region"]).size().unstack().to_string())
print("units with fewer than 100 spikes in total: %d; in the task epochs (reading B): %d; with no epoch over 100 (reading A): %d" % ((U.n_total < 100).sum(), (~U.active_B).sum(), (~U.active_A).sum()))

**Result: partially reproduced.** The dandiset holds 915 CA1 and 508 PFC units, not the 1,309 and 717 recorded: it holds the active units. Read that way, the deposited totals and the active counts match the paper's 934 and 508 (active by the two readings: CA1 871 and 876, PFC 493 and 494, all within tolerance). The later counts do not: odor-period active pyramidal cells are 138 to 146 in CA1 and 166 to 194 in PFC (expected 170 and 234), task-responsive pyramidal cells 111 to 122 and 124 to 156 (expected 138 and 185), under every reading. Without spike widths the rate-only rule gives 121 to 134 CA1 interneurons (expected 121) and 64 to 97 PFC interneurons (expected 44).

## fig3-choice-selective-fraction

> About a third of task-responsive pyramidal cells were choice selective, 47 of 138 in CA1 and 59
> of 185 in PFC.

SI from the per-trial rates of the sampling period on correct trials, against 1000 shuffles of the
odor labels; selective when 1.5 s.d. or more from the null mean (criterion). Under each reading of
unit-counts (rate threshold 7 Hz, the Methods' value; the code's thresholds as variants).

In [ ]:
rng = np.random.default_rng(3)
sess = {s.name: s for s in S}


def trial_rates(s, u):
    tr = s.trials
    spk = s.units.spikes[u]
    return opa.count_in(spk, tr.start.values, tr.stop.values) / tr.dur.values


def si_fast(rates, labels, n=1000, rng=rng):
    """SI and the shuffle null, vectorized (same definition as opa.si / opa.choice_selective)."""
    L = labels == "L"
    obs = opa.si(rates, labels)
    P = np.array([rng.permutation(L) for _ in range(n)])
    r1 = (P * rates).sum(1) / P.sum(1)
    r2 = (~P * rates).sum(1) / (~P).sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        null = (r1 - r2) / (r1 + r2)
    null = null[np.isfinite(null)]
    sel = bool(np.isfinite(obs) and len(null) and abs(obs - null.mean()) >= 1.5 * null.std())
    return obs, sel


SIc, SIsel = {}, {}
for i, r in U.iterrows():
    s = sess[r["name"]]
    c = correct(s)
    rates = trial_rates(s, r.unit)
    lab = s.trials.odor.values
    m = c & (lab != "")
    SIc[i], SIsel[i] = si_fast(rates[m], lab[m])
U["si_c"] = pd.Series(SIc)
U["selective"] = pd.Series(SIsel)
rows = []
for active in ("active_A", "active_B"):
    for intcol in ("int7", "int85", "int95"):
        for resp in ("p_resp_a", "p_resp_b"):
            for reg, exp_n, exp_k in (("CA1", 138, 47), ("PFC", 185, 59)):
                d = U[(U.region == reg) & U[active] & ~U[intcol] & U.odor_active & (U[resp] < 0.05)]
                rows.append(dict(active=active[-1], rate_thr=intcol[3:], resp=resp[-1], region=reg, task_resp_pyr=len(d), selective=int(d.selective.sum()), pct=round(100 * d.selective.mean(), 1), expected=f"{exp_k}/{exp_n} ({100 * exp_k / exp_n:.0f}%)"))
fs3 = pd.DataFrame(rows)
print(fs3.to_string(index=False))
print("tolerances: CA1 34 +/- 3.4%, PFC 31 +/- 3.1%; counts 47 +/- 5, 59 +/- 6")

**Result: partially reproduced.** In CA1 33.3% to 35.1% of task-responsive pyramidal cells are choice selective under every reading (expected 34 +/- 3.4%). In PFC 35.9% to 41.1% are (expected 31 +/- 3.1%), outside tolerance under every reading. No reading reproduces the 138 and 185 task-responsive cells, so the counts are not judged (CA1 38 to 42, PFC 51 to 56).

## fig3ij-si-anticorrelated

> In choice-selective pyramidal cells the selectivity index on incorrect trials was anti-correlated
> with that on correct trials, so the cells code the upcoming choice and not the odor.

The cell sets below use the Methods' reading of the selection (active at least 100 spikes over the
task epochs, interneuron above 7 Hz) under both task-responsiveness readings. SI readings (a) the
sampling-period rates, (b) the code's 50 ms bins from 0.2 to 1 s after odor onset, empty bins giving
SI 0, the per-bin SI averaged. Cells without incorrect trials of both odors are dropped and counted.
The selection of choice-selective cells is the one of fig3-choice-selective-fraction, from reading
(a)'s SI. Spearman correlation (paper); Pearson as a variant.

In [ ]:
def binned_si(s, u, mask):
    tr = s.trials
    spk = s.units.spikes[u]
    edges = np.arange(0.2, 1.0001, 0.05)
    out = []
    for od in ("L", "R"):
        m = mask & (tr.odor.values == od)
        st = tr.start.values[m]
        c = np.array([opa.count_in(spk, st + a, st + a + 0.05) for a in edges[:-1]])  # (bins, trials)
        out.append(c.mean(1) / 0.05 if m.any() else np.full(len(edges) - 1, np.nan))
    r1, r2 = out
    with np.errstate(invalid="ignore", divide="ignore"):
        v = np.where(r1 + r2 > 0, (r1 - r2) / (r1 + r2), 0.0)
    return float(np.mean(v))


def si_incorrect(i, reading):
    r = U.loc[i]
    s = sess[r["name"]]
    lab = s.trials.odor.values
    inc = ~correct(s) & (lab != "")
    cor = correct(s) & (lab != "")
    if (lab[inc] == "L").sum() < 1 or (lab[inc] == "R").sum() < 1:
        return np.nan, np.nan
    if reading == "a":
        rates = trial_rates(s, r.unit)
        return opa.si(rates[cor], lab[cor]), opa.si(rates[inc], lab[inc])
    return binned_si(s, r.unit, cor), binned_si(s, r.unit, inc)


def cellset(region, kind, resp="p_resp_a", intcol="int7", active="active_B"):
    d = U[(U.region == region) & U[active] & U.odor_active & (U[resp] < 0.05)]
    return d[~d[intcol]] if kind == "pyr" else d[d[intcol]]


def si_corr_table(kind, selective, label):
    rows = []
    for resp in ("p_resp_a", "p_resp_b"):
        for reg in ("CA1", "PFC"):
            d = cellset(reg, kind, resp)
            d = d[d.selective == selective]
            for rd in ("a", "b"):
                v = np.array([si_incorrect(i, rd) for i in d.index])
                ok = np.isfinite(v).all(1)
                rs = stats.spearmanr(v[ok, 0], v[ok, 1]) if ok.sum() > 2 else (np.nan, np.nan)
                rp = stats.pearsonr(v[ok, 0], v[ok, 1]) if ok.sum() > 2 else (np.nan, np.nan)
                rows.append(dict(cells=label, resp=resp[-1], region=reg, si_reading=rd, n=len(d), dropped=int((~ok).sum()), spearman_r=round(float(rs[0]), 3), spearman_p=float(rs[1]), pearson_r=round(float(rp[0]), 3), pearson_p=float(rp[1])))
    return pd.DataFrame(rows)

In [ ]:
t3ij = si_corr_table("pyr", True, "selective pyramidal")
print(t3ij.to_string(index=False))
print("paper: CA1 r = -0.58 (n = 47, p = 1.82e-5), PFC r = -0.68 (n = 59, p = 3.47e-9); tolerances 0.058 and 0.068")

**Result: partially reproduced.** Under both SI readings the correct and incorrect SIs of choice-selective pyramidal cells are anti-correlated in both regions (Spearman CA1 r = -0.49, p = 0.0019, and r = -0.59, p = 1.0e-4; PFC r = -0.54, p = 6.0e-5, and r = -0.55, p = 3.6e-5; 38 and 51 cells). The PFC correlations are weaker than -0.68 +/- 0.068 under both readings, and the CA1 one is within tolerance only with the binned SI.

## fig3s1e-nonselective-si

> The Results say the correct-incorrect anti-correlation was not significant among task-responsive
> cells that were not choice selective, while the legend reports a significant anti-correlation
> (r = -0.34 in both regions).

In [ ]:
t3s1e = si_corr_table("pyr", False, "non-selective pyramidal")
print(t3s1e.to_string(index=False))
print("readings: (a) Results, Spearman p >= 0.05; (b) legend, r = -0.34 +/- 0.034 with p < 0.05 (CA1 n = 91, PFC n = 126)")

**Result: blocked (methods underspecified).** The two SI readings give different outcomes. With sampling-period rates the non-selective cells show no significant correlation (CA1 r = -0.21, p = 0.078; PFC r = 0.01, p = 0.91), which matches the Results. With the code's binned SI both regions are anti-correlated (CA1 r = -0.31, p = 0.0059, within the legend's -0.34 +/- 0.034; PFC r = -0.44, p = 7.3e-5, outside it), which matches the legend in CA1 only. The paper's own inconsistency (Results against legend) is recorded as a finding.

## fig3s1c-odor-vs-reward-rate

> Firing rates of task-responsive pyramidal cells differed between odor sampling and reward
> consumption, in CA1 and in PFC.

Per cell the mean rate over the sampling periods and over the reward-well periods (entry to exit)
of the rewarded trials; the files have reward times only for rewarded trials, so the criterion's
reading "all completed trials" is the same set. Signed-rank across cells.

In [ ]:
rows = []
for resp in ("p_resp_a", "p_resp_b"):
    for reg in ("CA1", "PFC"):
        d = cellset(reg, "pyr", resp)
        od, rw = [], []
        for i, r in d.iterrows():
            s = sess[r["name"]]
            tr = s.trials
            c = correct(s) & np.isfinite(tr.rend.values)
            spk = s.units.spikes[r.unit]
            od.append(opa.count_in(spk, tr.start.values[c], tr.stop.values[c]).sum() / tr.dur.values[c].sum())
            rw.append(opa.count_in(spk, tr.rstart.values[c], tr.rend.values[c]).sum() / (tr.rend.values[c] - tr.rstart.values[c]).sum())
        od, rw = np.array(od), np.array(rw)
        rows.append(dict(resp=resp[-1], region=reg, n=len(d), odor_median_hz=round(float(np.median(od)), 2), reward_median_hz=round(float(np.median(rw)), 2), odor_higher=int((od > rw).sum()), p=float(stats.wilcoxon(od, rw).pvalue)))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: CA1 n = 138, p = 5.1e-13; PFC n = 185, p = 0.009")

**Result: reproduced.** Task-responsive pyramidal cells fire differently during odor sampling and at the reward well in both regions: CA1 median 5.4 against 0.7 Hz (101 of 114 cells higher during sampling, p = 1.3e-17), PFC 3.3 against 2.9 Hz (86 of 131, p = 8.1e-5); reading (b) gives the same.

## fig4c-interneuron-si

> CA1 choice-selective interneurons also had selectivity on incorrect trials anti-correlated with
> that on correct trials; PFC interneurons, few in number, did not.

Interneurons by rate alone (no waveforms; data check), task responsive and choice selective by the
same criteria as pyramidal cells. The code's 9.5 Hz threshold is shown as a variant.

In [ ]:
t4c = si_corr_table("int", True, "selective interneurons")
print(t4c.to_string(index=False))
rows = []
for intcol in ("int7", "int85", "int95"):
    for resp in ("p_resp_a", "p_resp_b"):
        for reg in ("CA1", "PFC"):
            d = cellset(reg, "int", resp, intcol)
            rows.append(dict(rate_thr=intcol[3:], resp=resp[-1], region=reg, task_resp_int=len(d), selective=int(d.selective.sum())))
print(pd.DataFrame(rows).to_string(index=False))
d95 = []
for reg in ("CA1", "PFC"):
    d = cellset(reg, "int", "p_resp_a", "int95")
    d = d[d.selective]
    v = np.array([si_incorrect(i, "a") for i in d.index])
    ok = np.isfinite(v).all(1)
    rs = stats.spearmanr(v[ok, 0], v[ok, 1])
    d95.append(f"{reg}: n = {ok.sum()}, r = {rs[0]:.3f}, p = {rs[1]:.3g}")
print("variant, 9.5 Hz, reading a:", "; ".join(d95))
print("paper: CA1 r(32) = -0.80, p = 3.4e-8 (tolerance 0.08); PFC r(10) = -0.38, p = 0.278; expected 32 +/- 3 and 10 +/- 1 selective interneurons")

**Result: not reproduced.** With interneurons defined by rate alone (no waveforms), CA1 has 39 selective task-responsive interneurons and PFC 23 (expected 32 and 10). The CA1 correlation is not significant with sampling-period rates (Spearman r = -0.19, p = 0.26) and is -0.40 (p = 0.011) with the binned SI, far from -0.80 +/- 0.08 under both; PFC is not significant with rates (p = 0.14) and is with the binned SI (p = 0.025). The criterion holds under neither reading.

## fig4d-ccg-lags

> Among CA1-PFC pairs with significant cross-correlation peaks during odor sampling, PFC
> interneurons tended to lead CA1 interneurons and CA1 pyramidal cells, CA1 interneuron-PFC
> pyramidal pairs were skewed too, and pyramidal-pyramidal pairs showed no skew.

Method from the criterion (Methods, Corrected spike cross-correlogram): simultaneously recorded
task-responsive CA1-PFC pairs; spikes of both cells in the sampling periods of correct and
incorrect trials; the cross-correlogram of PFC spike times relative to CA1 spikes from -0.15 to
0.15 s in 2.5 ms bins (positive lag: CA1 leads); the PFC cell's spikes jittered uniformly within
+/-50 ms 1000 times; the pointwise z-score against the jitter; the highest z within +/-100 ms
taken as the peak, the pair kept when it is significant. Readings: spikes from (a) whole sampling
periods, (b) the last 0.5 s; significance one-sided z > 1.645 or two-sided |z| > 1.96 (the peak
is the highest positive z either way). The code's z > 3 is a variant on the same jitters (its 200
jitters are not run). Cell types and task responsiveness as above (reading a, 7 Hz).

In [ ]:
EDGES = np.arange(-0.15, 0.15001, 0.0025)
CENT = (EDGES[:-1] + EDGES[1:]) / 2
NB = len(CENT)
rng4 = np.random.default_rng(44)


def window_spikes(spk, a, b):
    i0 = np.searchsorted(spk, a)
    i1 = np.searchsorted(spk, b)
    return np.concatenate([spk[x:y] for x, y in zip(i0, i1)]) if len(a) else np.zeros(0)


def ccg_peak(A, B, n_jit=1000, rng=rng4):
    """Lag and z of the highest jitter-corrected CCG peak of B relative to A within +/-100 ms."""
    if len(A) == 0 or len(B) == 0:
        return np.nan, np.nan
    lo = np.searchsorted(B, A - 0.2)
    hi = np.searchsorted(B, A + 0.2)
    cnt = hi - lo
    if cnt.sum() == 0:
        return np.nan, np.nan
    bidx = np.concatenate([np.arange(x, y) for x, y in zip(lo, hi)])
    d = B[bidx] - np.repeat(A, cnt)
    real = np.histogram(d, EDGES)[0].astype(float)
    null = np.zeros((n_jit, NB))
    for c0 in range(0, n_jit, 250):
        k = min(250, n_jit - c0)
        J = rng.uniform(-0.05, 0.05, (k, len(B))).astype(np.float32)
        dd = d[None, :].astype(np.float32) + J[:, bidx]
        b = np.floor((dd + 0.15) / 0.0025).astype(np.int64)
        ok = (b >= 0) & (b < NB)
        rows_ = np.repeat(np.arange(k), dd.shape[1]).reshape(k, -1)
        null[c0 : c0 + k] = np.bincount((rows_[ok] * NB + b[ok]), minlength=k * NB).reshape(k, NB)
    sd = null.std(0)
    z = np.where(sd > 0, (real - null.mean(0)) / np.where(sd > 0, sd, 1), 0.0)
    w = np.abs(CENT) <= 0.1
    k = np.argmax(np.where(w, z, -np.inf))
    return CENT[k], z[k]


t0 = time.time()
U["ctype"] = np.where(U.int7, "int", "pyr")
resp = U[U.active_B & U.odor_active & (U.p_resp_a < 0.05)]
pair_rows = []
for s in S:
    tr = s.trials
    ca = resp[(resp["name"] == s.name) & (resp.region == "CA1")]
    pf = resp[(resp["name"] == s.name) & (resp.region == "PFC")]
    if not len(ca) or not len(pf):
        continue
    for win, (a, b) in (("a", (tr.start.values, tr.stop.values)), ("b", (np.maximum(tr.stop.values - 0.5, tr.start.values), tr.stop.values))):
        A_sp = {i: window_spikes(s.units.spikes[r.unit], a, b) for i, r in ca.iterrows()}
        B_sp = {i: window_spikes(s.units.spikes[r.unit], a, b) for i, r in pf.iterrows()}
        for i, r1 in ca.iterrows():
            for j, r2 in pf.iterrows():
                lag, z = ccg_peak(A_sp[i], B_sp[j])
                pair_rows.append(dict(name=s.name, subject=s.subject, win=win, ca1=i, pfc=j, type=f"CA1 {r1.ctype}-PFC {r2.ctype}", lag=lag, z=z))
pairs = pd.DataFrame(pair_rows)
print(f"pairs tested per window: {len(pairs) // 2}; time {time.time() - t0:.0f} s")

In [ ]:
rows = []
exp_n = {"CA1 int-PFC int": 87, "CA1 int-PFC pyr": 283, "CA1 pyr-PFC int": 111, "CA1 pyr-PFC pyr": 224}
for win in ("a", "b"):
    for thr_name, thr in (("one-sided z>1.645", 1.645), ("two-sided |z|>1.96", 1.96), ("variant z>3", 3.0)):
        for ty in exp_n:
            d = pairs[(pairs.win == win) & (pairs.type == ty) & (pairs.z > thr)]
            lags = d.lag.values
            p = stats.wilcoxon(lags).pvalue if len(lags) > 5 and (lags != 0).any() else np.nan
            rows.append(dict(window=win, threshold=thr_name, type=ty, tested=int(((pairs.win == win) & (pairs.type == ty)).sum()), significant=len(lags), expected_n=exp_n[ty], median_lag_ms=round(1000 * float(np.median(lags)), 2) if len(lags) else np.nan, mean_lag_ms=round(1000 * float(np.mean(lags)), 2) if len(lags) else np.nan, p_signrank=p))
ccg = pd.DataFrame(rows)
print(ccg.to_string(index=False))
print("paper: int-int x = -28 ms, p = 3.5e-8; CA1 int-PFC pyr p = 2.11e-3; CA1 pyr-PFC int x = -13 ms, p = 3.4e-3; pyr-pyr p = 0.14")

**Result: not reproduced.** Under every reading (whole sampling period or last 0.5 s; one-sided or two-sided threshold; the code's z > 3 as a variant) no pair type shows a lag skew: interneuron pairs median lag -6.3 to 5.0 ms, p >= 0.21; CA1 pyramidal-PFC interneuron pairs p >= 0.23; CA1 interneuron-PFC pyramidal pairs p >= 0.23; pyramidal pairs p >= 0.57. Almost every tested pair has a significant peak at the pointwise thresholds (for example 196 of 203 interneuron pairs), against the paper's 87.

## Phase locking (fig5)

Spike phases come from `opa.lfp_measures`: the ground-referenced LFP of the tetrode with most cells
in each region that day (OB: the first OB tetrode), band-passed at 20 to 30 Hz (beta) or 7 to 8 Hz
(RR) with a zero-phase Butterworth of order 2 (the order is not given; an assumption), Hilbert
phase with 0 at the peak, spikes pooled over the sampling periods of correct trials (criterion).
A cell is locked when the Rayleigh test gives p < 0.05, with readings for the minimum spike count:
more than 10 spikes (Path A) or more than 5 (Path B). Cells are the task-responsive cells (reading
a, 7 Hz; reading b of task responsiveness shown for the counts).

In [ ]:
def unit_phases(i, lfp_region, band, trials_mask=None):
    r = U.loc[i]
    s = sess[r["name"]]
    rec = M[s.name]["spike_phase"][r.unit]
    if lfp_region not in rec:
        return np.zeros(0)
    pb, pr, tri = rec[lfp_region]
    ph = pb if band == "beta" else pr
    m = correct(s)[tri] if trials_mask is None else trials_mask(s)[tri]
    return ph[m]


LOCAL = {"CA1": "CA1", "PFC": "PFC"}
lock_rows = []
for i, r in U[U.active_B & U.odor_active & ((U.p_resp_a < 0.05) | (U.p_resp_b < 0.05))].iterrows():
    row = dict(idx=i, region=r.region, ctype=r.ctype, resp_a=r.p_resp_a < 0.05, resp_b=r.p_resp_b < 0.05, selective=bool(r.selective))
    for band in ("beta", "rr"):
        for lr in ("CA1", "PFC", "OB"):
            ph = unit_phases(i, lr, band)
            p, z, mvl = opa.rayleigh(ph)
            row[f"n_{lr}_{band}"] = len(ph)
            row[f"p_{lr}_{band}"] = p
            row[f"z_{lr}_{band}"] = z
            row[f"mu_{lr}_{band}"] = float(np.angle(np.mean(np.exp(1j * ph)))) if len(ph) else np.nan
    lock_rows.append(row)
L = pd.DataFrame(lock_rows).set_index("idx")
for band in ("beta", "rr"):
    L[f"p_local_{band}"] = [L.loc[i, f"p_{r}_{band}"] for i, r in zip(L.index, L.region)]
    L[f"n_local_{band}"] = [L.loc[i, f"n_{r}_{band}"] for i, r in zip(L.index, L.region)]
    L[f"z_local_{band}"] = [L.loc[i, f"z_{r}_{band}"] for i, r in zip(L.index, L.region)]
    L[f"mu_local_{band}"] = [L.loc[i, f"mu_{r}_{band}"] for i, r in zip(L.index, L.region)]


def locked(df, band, minspk, where="local", alpha=0.05):
    return (df[f"p_{where}_{band}"] < alpha) & (df[f"n_{where}_{band}"] > minspk)


LA = L[L.resp_a]
print("task-responsive cells (reading a):", LA.groupby(["region", "ctype"]).size().to_dict())

### fig5c-local-locking

> Most CA1 task-responsive cells and a minority of PFC cells were locked to local RR; a modest share
> of CA1 pyramidal cells and a large share of CA1 interneurons were locked to local beta, while PFC
> cells were locked to beta no more than chance.

Chance is 0.05; binomial test, one-sided, of the number locked against n and 0.05 (criterion).

In [ ]:
exp5c = {("CA1", "pyr", "rr"): (113, 138), ("CA1", "int", "rr"): (74, 83), ("PFC", "pyr", "rr"): (42, 185), ("PFC", "int", "rr"): (12, 28), ("CA1", "pyr", "beta"): (28, 138), ("CA1", "int", "beta"): (36, 83), ("PFC", "pyr", "beta"): (8, 185), ("PFC", "int", "beta"): (2, 28)}
rows = []
for respcol in ("resp_a", "resp_b"):
    for minspk in (10, 5):
        for (reg, ct, band), (k_exp, n_exp) in exp5c.items():
            d = L[L[respcol] & (L.region == reg) & (L.ctype == ct)]
            k = int(locked(d, band, minspk).sum())
            n = len(d)
            rows.append(dict(resp=respcol[-1], min_spikes=f">{minspk}", region=reg, type=ct, band=band, n=n, locked=k, pct=round(100 * k / n, 1), expected=f"{k_exp}/{n_exp} ({100 * k_exp / n_exp:.0f}%)", p_vs_chance=float(stats.binomtest(k, n, 0.05, alternative="greater").pvalue)))
t5c = pd.DataFrame(rows)
print(t5c.to_string(index=False))
print("interneuron denominators in the legend: CA1 96 (RR 74/96 = 77%, beta 36/96 = 38%), PFC 36 (12/36 = 33%, 2/36 = 6%)")
print("\nper animal, reading a, > 10 spikes, fraction locked:")
d = LA.assign(subject=[U.loc[i, "subject"] for i in LA.index])
d["beta"] = locked(d, "beta", 10)
d["rr"] = locked(d, "rr", 10)
print(d.groupby(["region", "ctype", "subject"])[["beta", "rr"]].mean().round(2).unstack(0).to_string())

**Result: partially reproduced.** The significance pattern holds under both spike-count readings: 19.3% of CA1 pyramidal cells lock to local beta (p = 4.9e-8 against 5%), while PFC pyramidal cells (4.6%, p = 0.64) and interneurons (7.6%, p = 0.20) do not exceed chance. Some fractions are within tolerance (CA1 pyramidal RR 75.4% against 82 +/- 8%; CA1 pyramidal beta 19.3% against 20 +/- 2%; CA1 interneuron beta 44.6% against 43 +/- 5%; PFC beta 4.6% and 7.6%) and some are not (CA1 interneuron RR 71.4% against 89% of 83 or 77% of 96; PFC pyramidal RR 19.8% against 22.7 +/- 2.2%; PFC interneuron RR 29.1% against 43% or 33%).

### fig5d-cross-region-beta

> More CA1 interneurons were locked to PFC beta than chance, while CA1 pyramidal cells and PFC
> cells showed no cross-regional beta locking beyond chance.

Each task-responsive cell is tested against beta on the CA1, PFC and OB main tetrodes, locked to a
region when the Rayleigh p is below 0.05/3, and assigned to the region with the smallest p
(criterion). Binomial test of each cross-region count against chance, readings 0.0167 and 0.05.

In [ ]:
rows = []
for minspk in (10, 5):
    for reg, ct, target, k_exp, n_exp in (("CA1", "int", "PFC", 9, 83), ("CA1", "int", "OB", 2, 83), ("CA1", "pyr", "PFC", 11, 138), ("CA1", "pyr", "OB", 10, 138), ("PFC", "pyr", "CA1", 10, 185), ("PFC", "pyr", "OB", None, 185), ("PFC", "int", "CA1", None, 28), ("PFC", "int", "OB", None, 28)):
        d = LA[(LA.region == reg) & (LA.ctype == ct)]
        P = np.stack([np.where(d[f"n_{x}_beta"] > minspk, d[f"p_{x}_beta"], 1.0) for x in ("CA1", "PFC", "OB")], 1)
        best = np.array(["CA1", "PFC", "OB"])[np.argmin(P, 1)]
        k = int(((P.min(1) < 0.05 / 3) & (best == target)).sum())
        n = len(d)
        rows.append(dict(min_spikes=f">{minspk}", cells=f"{reg} {ct}", to=f"{target} beta", n=n, count=k, expected=f"{k_exp}/{n_exp}" if k_exp is not None else "", p_chance_0167=float(stats.binomtest(k, n, 0.05 / 3, alternative="greater").pvalue), p_chance_05=float(stats.binomtest(k, n, 0.05, alternative="greater").pvalue)))
t5d = pd.DataFrame(rows)
print(t5d.to_string(index=False))

**Result: reproduced.** 12 of 112 CA1 interneurons lock maximally to PFC beta, above chance at either chance level (p = 4.3e-7 at 0.0167, p = 0.010 at 0.05); CA1 pyramidal cells to PFC or OB beta (3 of 114 each, p = 0.30) and PFC cells to CA1 or OB beta (PFC pyramidal to CA1 6 of 131, p = 0.023 at 0.0167, above the 0.0167 threshold) do not. The counts differ from the paper's (12 against 9, 3 against 11 and 10, 6 against 10).

### fig5gh-phase-preference

> Preferred beta and RR phases were consistent across task-responsive CA1 cells, while in PFC only
> interneurons had a consistent preferred RR phase.

Each task-responsive cell's preferred phase is the circular mean of its correct-trial spike phases
on the local tetrode, locked or not (criterion; the von Mises maximum-likelihood mean is the
circular mean). Rayleigh test across cells. Cells without spikes are dropped.

In [ ]:
rows = []
exp_sig = {("CA1", "pyr", "beta"): True, ("CA1", "pyr", "rr"): True, ("CA1", "int", "beta"): True, ("CA1", "int", "rr"): True, ("PFC", "int", "rr"): True, ("PFC", "pyr", "beta"): False, ("PFC", "pyr", "rr"): False, ("PFC", "int", "beta"): False}
for respcol in ("resp_a", "resp_b"):
    for (reg, ct, band), sig in exp_sig.items():
        d = L[L[respcol] & (L.region == reg) & (L.ctype == ct)]
        mu = d[f"mu_local_{band}"].dropna().values
        p, z, r = opa.rayleigh(mu)
        rows.append(dict(resp=respcol[-1], region=reg, type=ct, band=band, cells=len(mu), mean_phase_deg=round(float(np.degrees(np.angle(np.mean(np.exp(1j * mu))))), 1), R=round(r, 3), p=p, paper_significant=sig, holds=(p < 0.05) == sig))
t5gh = pd.DataFrame(rows)
print(t5gh.to_string(index=False))

**Result: partially reproduced.** Seven of the eight tests hold under both selections: preferred phases are consistent across CA1 pyramidal cells (beta p = 2.1e-5, RR p = 0.0055) and interneurons (p = 1.9e-5, 8.7e-8) and PFC interneurons for RR (p = 0.011), and not across PFC pyramidal cells (beta p = 0.30, RR p = 0.86). PFC interneurons also have a consistent beta phase (p = 0.0063), where the paper finds none.

### fig5-rr-more-than-beta

> More cells were phase locked to RR than to beta, and the depth of RR modulation was stronger, in
> CA1 and in PFC.

Proportions of task-responsive cells locked to RR and to beta, a pooled z-test for two proportions,
two-sided and one-sided; readings local LFP only, and CA1 or PFC LFP (the code). Rayleigh Z of each
task-responsive pyramidal cell for RR against beta by rank-sum (legend).

In [ ]:
def ztest_prop(k1, n1, k2, n2):
    p = (k1 + k2) / (n1 + n2)
    se = np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = (k1 / n1 - k2 / n2) / se
    return z, 2 * stats.norm.sf(abs(z)), stats.norm.sf(z)


rows = []
for minspk in (10, 5):
    for where in ("local", "CA1 or PFC"):
        for reg in ("CA1", "PFC"):
            d = LA[LA.region == reg]
            if where == "local":
                kr, kb = locked(d, "rr", minspk).sum(), locked(d, "beta", minspk).sum()
            else:
                kr = (locked(d, "rr", minspk, "CA1") | locked(d, "rr", minspk, "PFC")).sum()
                kb = (locked(d, "beta", minspk, "CA1") | locked(d, "beta", minspk, "PFC")).sum()
            n = len(d)
            z, p2, p1 = ztest_prop(kr, n, kb, n)
            pyr = d[d.ctype == "pyr"]
            zr, zb = pyr["z_local_rr"].dropna(), pyr["z_local_beta"].dropna()
            rows.append(dict(min_spikes=f">{minspk}", lfp=where, region=reg, n=n, rr_locked=int(kr), beta_locked=int(kb), z=round(float(z), 2), p_two_sided=p2, p_one_sided=p1, rayleighZ_rr_median=round(float(zr.median()), 2), rayleighZ_beta_median=round(float(zb.median()), 2), p_ranksum_Z=float(stats.ranksums(zr, zb).pvalue)))
t5rr = pd.DataFrame(rows)
print(t5rr.to_string(index=False))
print("paper: proportions CA1 p = 6.8e-23, PFC p = 8.8e-9; Rayleigh Z RR vs beta CA1 p = 3.2e-26 (n = 138), PFC p = 0.002 (n = 185)")

**Result: partially reproduced.** Under every reading more task-responsive cells lock to RR than to beta (CA1 166 against 72 of 226, PFC 49 against 12 of 210 on the local LFP; z-test p = 8.3e-19 and 3.0e-7). The Rayleigh Z of pyramidal cells is larger for RR in CA1 (median 13.5 against 0.95, p = 2.9e-20) but not significantly in PFC (0.89 against 0.77, p = 0.091).

### fig5s1de-beta-rr-overlap

> The share of cells locked to both beta and RR did not differ from that expected by chance from
> the share locked to each, in CA1 and PFC, for pyramidal cells and interneurons.

Chance probability the product of the fractions locked to beta and to RR; binomial test of the
number locked to both, two-sided (criterion). Local LFP.

In [ ]:
rows = []
for minspk in (10, 5):
    for reg in ("CA1", "PFC"):
        for ct in ("pyr", "int"):
            d = LA[(LA.region == reg) & (LA.ctype == ct)]
            b, r_ = locked(d, "beta", minspk), locked(d, "rr", minspk)
            n = len(d)
            pc = b.mean() * r_.mean()
            k = int((b & r_).sum())
            rows.append(dict(min_spikes=f">{minspk}", region=reg, type=ct, n=n, beta=int(b.sum()), rr=int(r_.sum()), both=k, observed=round(k / n, 3), chance=round(pc, 3), p=float(stats.binomtest(k, n, pc).pvalue)))
t5de = pd.DataFrame(rows)
print(t5de.to_string(index=False))
print("paper: pyramidal CA1 p = 0.08, PFC 0.06; interneurons CA1 0.11, PFC 0.20")

**Result: reproduced.** The share of cells locked to both rhythms does not differ from the product of the two shares in any group (CA1 pyramidal 0.167 against 0.146, p = 0.51; CA1 interneurons 0.295 against 0.319, p = 0.61; PFC pyramidal p = 1.0; PFC interneurons p = 0.69).

### fig5s1fh-selective-and-locked

> Choice-selective cells were no more or less likely than chance to be phase locked to beta or to
> RR, in CA1 and PFC.

Chance the product of the fractions choice selective and phase locked among task-responsive cells;
binomial test, two-sided. "Phase locked" is to any region's rhythm (CA1, PFC or OB tetrode), with
local only as a reading (criterion).

In [ ]:
rows = []
for minspk in (10, 5):
    for where in ("any region", "local"):
        for ct, band, reg in (("pyr", "beta", "CA1"), ("pyr", "beta", "PFC"), ("int", "beta", "CA1"), ("int", "beta", "PFC"), ("pyr", "rr", "CA1"), ("pyr", "rr", "PFC")):
            d = LA[(LA.region == reg) & (LA.ctype == ct)]
            if where == "local":
                lk = locked(d, band, minspk)
            else:
                lk = locked(d, band, minspk, "CA1") | locked(d, band, minspk, "PFC") | locked(d, band, minspk, "OB")
            sel = d.selective.astype(bool)
            n = len(d)
            pc = sel.mean() * lk.mean()
            k = int((sel & lk).sum())
            rows.append(dict(min_spikes=f">{minspk}", locked_to=where, type=ct, band=band, region=reg, n=n, selective=int(sel.sum()), locked=int(lk.sum()), both=k, chance=round(pc, 3), p=float(stats.binomtest(k, n, pc).pvalue)))
t5fh = pd.DataFrame(rows)
print(t5fh.to_string(index=False))
print("paper: all p > 0.05")

**Result: reproduced.** None of the six binomial tests finds more or fewer selective and locked cells than chance, with locking to any region (p from 0.54 to 1.0) or to the local LFP (p from 0.45 to 1.0).

### fig5e-beta-mvl-correct-incorrect, fig5f-rr-mvl-no-difference, fig5s2-ca1-int-pfc-beta-accuracy

Method from the Methods (Phase locking): correct-trial spikes subsampled to the number of
incorrect-trial spikes, the mean vector length of the subsample bootstrapped 1000 times, the mean
bootstrapped MVL compared with the incorrect-trial MVL; signed-rank test across cells. Cells
without incorrect-trial spikes are dropped and counted; cells with more incorrect than correct
spikes keep all correct spikes (subsampling cannot reach the count; a standard choice).

In [ ]:
def mvl_pair(i, lfp_region, band, n_boot=1000, seed=55):
    c = unit_phases(i, lfp_region, band, correct)
    w = unit_phases(i, lfp_region, band, lambda s: ~correct(s))
    if len(w) < 2 or len(c) < 2:
        return np.nan, np.nan
    k = min(len(w), len(c))
    rng5 = np.random.default_rng([seed, int(i)])  # one generator per cell, so that a cell's value does not depend on the others
    idx = np.array([rng5.choice(len(c), k, replace=False) for _ in range(n_boot)])
    boot = np.abs(np.exp(1j * c[idx]).mean(1)).mean()
    return boot, float(np.abs(np.mean(np.exp(1j * w))))


def mvl_test(cells, lfp_region_of, band, seed=55):
    v = np.array([mvl_pair(i, lfp_region_of(i), band, seed=seed) for i in cells])
    ok = np.isfinite(v).all(1) if len(v) else np.zeros(0, bool)
    if ok.sum() < 3:
        return dict(n=len(cells), used=int(ok.sum()), median_correct=np.nan, median_incorrect=np.nan, lower_incorrect=0, p=np.nan)
    return dict(n=len(cells), used=int(ok.sum()), median_correct=round(float(np.median(v[ok, 0])), 3), median_incorrect=round(float(np.median(v[ok, 1])), 3), lower_incorrect=int((v[ok, 1] < v[ok, 0]).sum()), p=float(stats.wilcoxon(v[ok, 0], v[ok, 1]).pvalue))

#### fig5e-beta-mvl-correct-incorrect

> Beta-locked CA1 pyramidal cells and interneurons and PFC pyramidal cells were more strongly
> locked to local beta on correct than on incorrect trials.

In [ ]:
rows = []
for minspk in (10, 5):
    for reg, ct in (("CA1", "pyr"), ("CA1", "int"), ("PFC", "pyr"), ("PFC", "int")):
        d = LA[(LA.region == reg) & (LA.ctype == ct)]
        cells = d.index[locked(d, "beta", minspk)]
        rows.append(dict(min_spikes=f">{minspk}", group=f"{reg} {ct}", **mvl_test(cells, lambda i: U.loc[i, "region"], "beta")))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: CA1 pyr p = 0.014 (n = 28), CA1 int p = 0.038 (n = 36), PFC pyr p = 0.039 (n = 8)")
print("\nthe bootstrap is random: p over 10 other seeds, > 10 spikes")
for reg, ct in (("CA1", "pyr"), ("CA1", "int"), ("PFC", "pyr")):
    d = LA[(LA.region == reg) & (LA.ctype == ct)]
    cells = d.index[locked(d, "beta", 10)]
    ps = [mvl_test(cells, lambda i: U.loc[i, "region"], "beta", seed=100 + k)["p"] for k in range(10)]
    print(f"{reg} {ct}: p from {min(ps):.3f} to {max(ps):.3f}; under 0.05 in {sum(p < 0.05 for p in ps)} of 10")

**Result: reproduced.** Beta-locked cells have a lower MVL on incorrect trials in all three groups: CA1 pyramidal cells (13 of 22 lower, p = 0.039), CA1 interneurons (32 of 50, p = 0.014) and PFC pyramidal cells (6 of 6, p = 0.031). The CA1 pyramidal result is close to the threshold: over 10 other bootstrap seeds its p ranges from 0.033 to 0.054 and is under 0.05 in 8.

#### fig5f-rr-mvl-no-difference

> Locking to local RR did not differ between correct and incorrect trials.

In [ ]:
rows = []
for minspk in (10, 5):
    for reg, ct in (("CA1", "pyr"), ("CA1", "int"), ("PFC", "pyr"), ("PFC", "int")):
        d = LA[(LA.region == reg) & (LA.ctype == ct)]
        cells = d.index[locked(d, "rr", minspk)]
        rows.append(dict(min_spikes=f">{minspk}", group=f"{reg} {ct}", **mvl_test(cells, lambda i: U.loc[i, "region"], "rr")))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: no group significant")

**Result: reproduced.** No RR-locked group has a different MVL on correct and incorrect trials (CA1 pyramidal p = 0.88, CA1 interneurons p = 0.98, PFC pyramidal p = 0.13, PFC interneurons p = 0.34).

#### fig5s2-ca1-int-pfc-beta-accuracy

> CA1 interneurons locked to PFC beta were more strongly locked on correct than on incorrect trials.

Readings for which interneurons form the set: locked to PFC beta at p < 0.05, or at p < 0.0167;
the other cross-regional combinations are reported.

In [ ]:
rows = []
for minspk in (10, 5):
    for alpha in (0.05, 0.05 / 3):
        for reg, ct, target in (("CA1", "int", "PFC"), ("CA1", "int", "OB"), ("CA1", "pyr", "PFC"), ("CA1", "pyr", "OB"), ("PFC", "pyr", "CA1"), ("PFC", "int", "CA1")):
            d = LA[(LA.region == reg) & (LA.ctype == ct)]
            cells = d.index[locked(d, "beta", minspk, target, alpha)]
            rows.append(dict(min_spikes=f">{minspk}", alpha=round(alpha, 4), group=f"{reg} {ct} to {target} beta", **mvl_test(cells, lambda i, t=target: t, "beta")))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: CA1 interneurons to PFC beta n = 20, p = 0.0023 (Results) or 0.0031 (legend); other combinations p > 0.05")

**Result: reproduced.** CA1 interneurons locked to PFC beta are more strongly locked on correct trials under both readings (p < 0.05: 33 cells, 26 lower on incorrect trials, p = 1.0e-4; p < 0.0167: 22 cells, p = 1.5e-4). Of the other combinations, CA1 interneurons locked to OB beta also differ (p = 0.0081 and 0.039), where the legend gives all p > 0.05.

## fig6d-discrimination-time

> CA1 and PFC ensemble trajectories for the two choices separated within the odor sampling period,
> at similar times (CA1 200 to 800 ms, PFC 400 to 700 ms after odor onset).

Method from the criterion (Methods, PCA; code): sessions with at least 4 task-responsive pyramidal
cells in the region; spike counts in 100 ms bins from 0 to 1 s after odor onset, correct trials,
as rates; first 3 PCs; the Euclidean distance between the mean left-bound and right-bound
trajectories; a null from 1000 shuffles of the trial labels; the discrimination time is the first
bin whose distance exceeds the null's bound. Readings: PCA fitted (c) on the trial-averaged bins by
neurons matrix (Methods) or (d) per bin on single trials (code); the bound the 95th or the 97.5th
percentile. A bin's time is its end (100 ms for the first bin); its start is 100 ms earlier.

In [ ]:
def pca_proj(X, ncomp=3):
    Xc = X - X.mean(0)
    u, sv, vt = np.linalg.svd(Xc, full_matrices=False)
    return vt[: min(ncomp, vt.shape[0])].T, X.mean(0)


def traj_dist(R, lab, mode):
    """R: trials x bins x neurons rates; lab boolean (left). Distance per bin between the mean
    trajectories of the two groups in the first 3 PCs."""
    if mode == "c":
        avg = R.mean(0)  # bins x neurons
        W, mu = pca_proj(avg)
        a = (R[lab].mean(0) - mu) @ W
        b = (R[~lab].mean(0) - mu) @ W
        return np.linalg.norm(a - b, axis=1)
    out = []
    for t in range(R.shape[1]):
        W, mu = pca_proj(R[:, t, :])
        P = (R[:, t, :] - mu) @ W
        out.append(np.linalg.norm(P[lab].mean(0) - P[~lab].mean(0)))
    return np.array(out)


rng6 = np.random.default_rng(66)
BINS = np.arange(0, 1.0001, 0.1)
pca_rows = []
evenodd_rows = []
for s in S:
    tr = s.trials
    c = correct(s) & (tr.side.values != "")
    for reg in ("CA1", "PFC"):
        d = cellset(reg, "pyr")
        d = d[d["name"] == s.name]
        if len(d) < 4:
            continue
        st = tr.start.values[c]
        R = np.stack([np.stack([opa.count_in(s.units.spikes[u], st + BINS[k], st + BINS[k + 1]) for k in range(10)], 1) for u in d.unit], 2) / 0.1
        lab = tr.side.values[c] == "L"
        for mode in ("c", "d"):
            real = traj_dist(R, lab, mode)
            null = np.array([traj_dist(R, rng6.permutation(lab), mode) for _ in range(1000)])
            for q in (95, 97.5):
                bound = np.percentile(null, q, axis=0)
                cross = np.nonzero(real > bound)[0]
                pca_rows.append(dict(name=s.name, subject=s.subject, region=reg, cells=len(d), mode=mode, q=q, t_ms=int(round(1000 * BINS[cross[0] + 1])) if len(cross) else np.nan))
                # even-odd control: per odor, even against odd trials, averaged over the two odors
                eo = []
                for side in (True, False):
                    idx = np.nonzero(lab == side)[0]
                    sub = R[idx]
                    even = np.arange(len(idx)) % 2 == 0
                    eo.append(traj_dist(sub, even, mode))
                eo = np.mean(eo, 0)
                evenodd_rows.append(dict(name=s.name, region=reg, mode=mode, q=q, crosses=int((eo > bound).sum())))
pca = pd.DataFrame(pca_rows)
evo = pd.DataFrame(evenodd_rows)
print("sessions with at least 4 task-responsive pyramidal cells:", pca[(pca["mode"] == "c") & (pca.q == 95)].groupby("region").size().to_dict(), "; animals:", pca.groupby("region").subject.nunique().to_dict())

In [ ]:
rows = []
for mode in ("c", "d"):
    for q in (95, 97.5):
        d = pca[(pca["mode"] == mode) & (pca.q == q)]
        ca, pf = d[d.region == "CA1"], d[d.region == "PFC"]
        both = ca.merge(pf, on="name", suffixes=("_ca1", "_pfc")).dropna(subset=["t_ms_ca1", "t_ms_pfc"])
        p = stats.wilcoxon(both.t_ms_ca1, both.t_ms_pfc).pvalue if len(both) > 3 and (both.t_ms_ca1 != both.t_ms_pfc).any() else np.nan
        rows.append(dict(pca=mode, bound=q, ca1_sessions=len(ca), ca1_crossed=int(ca.t_ms.notna().sum()), ca1_range_ms=f"{ca.t_ms.min():.0f}-{ca.t_ms.max():.0f}", ca1_median=ca.t_ms.median(), pfc_sessions=len(pf), pfc_crossed=int(pf.t_ms.notna().sum()), pfc_range_ms=f"{pf.t_ms.min():.0f}-{pf.t_ms.max():.0f}", pfc_median=pf.t_ms.median(), paired_sessions=len(both), p_signrank=p))
print(pd.DataFrame(rows).to_string(index=False))
print("expected ranges CA1 200-800 ms (widened 180-880), PFC 400-700 ms (widened 360-770); signed-rank p >= 0.05 (paper 0.68, n = 24 sessions across 8 animals)")
print("\nper-session times, reading c, 95th percentile:")
print(pca[(pca["mode"] == "c") & (pca.q == 95)].pivot(index="name", columns="region", values="t_ms").to_string())

**Result: not reproduced.** Only 12 CA1 and 18 PFC sessions have at least 4 task-responsive pyramidal cells (the paper 24 sessions). Under every reading the discrimination times spread beyond the expected ranges (CA1 100 or 200 to 1000 ms against 200 to 800; PFC 100 to 900 or 1000 ms against 400 to 700). The CA1 and PFC times do not differ (p = 0.44 to 0.63, but on only 5 paired sessions).

## fig6c-even-odd-control

> A control distance between even and odd trials of the same odor never crossed significance in any
> session.

For each session and region of fig6d, the distance between the mean trajectories of even and odd
trials of each odor (correct trials, so odor and chosen side coincide), averaged over the two odors,
compared bin by bin with the same null bound (criterion).

In [ ]:
print(evo.groupby(["mode", "q", "region"]).apply(lambda d: f"{int((d.crosses > 0).sum())} of {len(d)} sessions cross in at least one bin", include_groups=False).to_string())

**Result: not reproduced.** The even-odd distance crosses the null bound in at least one bin in 5 to 8 of 12 CA1 sessions and 10 to 14 of 18 PFC sessions, depending on the reading.

## fig6e-glm-choice-prediction

> A GLM on task-responsive pyramidal ensembles predicted the upcoming choice before odor port exit,
> from 0.1 s before exit in CA1 and 0.3 s before in PFC.

Method from the criterion (Methods, GLM; code): task-responsive pyramidal cells active (at least
one spike in the window) on at least 10 trials; spike counts per cell as predictors; the chosen
side, all trials, as response; a binomial GLM with the code's logit link (the Methods' log link is
not run: with a log link a binomial model's predictions can leave 0 to 1); five-fold
cross-validation; a null from 100 shuffles of the choice labels per session, each with its own
folds; a rank-sum test over sessions of the real accuracy against the mean shuffled accuracy, per
window. Readings for the alignment: (a) windows from odor port exit minus 0.6 s to exit plus w, w
from -0.5 to 0.2 s, a window's time its end; (b) windows from odor onset of 0.1 to 1.0 s, a
window's time before exit read as 0.82 s minus its length. The reported time is the earliest window
from which all later windows are significant. A session enters a window when at least one cell
qualifies (the Methods give no minimum; an assumption, with at least 4 cells as a variant).

In [ ]:
def glm_windows(s, cells, reading):
    tr = s.trials
    ok = tr.side.values != ""
    y = (tr.side.values[ok] == "L").astype(float)
    out = []
    if reading == "a":
        wins = [(-0.6, w) for w in np.round(np.arange(-0.5, 0.21, 0.1), 1)]
        ref = tr.stop.values[ok]
    else:
        wins = [(0.0, w) for w in np.round(np.arange(0.1, 1.01, 0.1), 1)]
        ref = tr.start.values[ok]
    for a, b in wins:
        X = np.stack([opa.count_in(s.units.spikes[u], ref + a, ref + b) for u in cells], 1).astype(float)
        keep = (X > 0).sum(0) >= 10
        out.append(((a, b), X[:, keep], y))
    return out


def glm_session(s, cells, reading, rng, n_shuf=100):
    res = []
    for (a, b), X, y in glm_windows(s, cells, reading):
        if X.shape[1] == 0:
            res.append(dict(win=b, cells=0, real=np.nan, shuf=np.nan))
            continue
        real = opa.cv_accuracy(X, y, rng)
        shuf = np.mean([opa.cv_accuracy(X, rng.permutation(y), rng) for _ in range(n_shuf)])
        res.append(dict(win=b, cells=X.shape[1], real=real, shuf=shuf))
    return res


def glm_run(kind_sets, reading, seed, n_shuf=100):
    rng = np.random.default_rng(seed)
    rows = []
    for s in S:
        for reg in ("CA1", "PFC"):
            cells = kind_sets(s, reg)
            if not len(cells):
                continue
            for r in glm_session(s, cells, reading, rng, n_shuf):
                rows.append(dict(name=s.name, region=reg, **r))
    return pd.DataFrame(rows)


def glm_summary(df, reading, min_cells=1):
    rows = []
    for reg in ("CA1", "PFC"):
        d = df[(df.region == reg) & (df.cells >= min_cells)].dropna(subset=["real"])
        sig = []
        for w, g in d.groupby("win"):
            p = stats.ranksums(g.real, g.shuf).pvalue
            sig.append((w, len(g), g.real.mean(), g.shuf.mean(), p))
        sig = pd.DataFrame(sig, columns=["win", "sessions", "acc", "acc_shuf", "p"])
        ok = sig.p < 0.05
        first = None
        for k in range(len(sig)):
            if ok.iloc[k:].all():
                first = sig.win.iloc[k]
                break
        t_before = (-first if reading == "a" else round(0.82 - first, 2)) if first is not None else np.nan
        rows.append((reg, sig, first, t_before))
    return rows


def pyr_cells(s, reg, resp="p_resp_a"):
    d = cellset(reg, "pyr", resp)
    return list(d[d["name"] == s.name].unit)


t0 = time.time()
glm_pyr = {rd: glm_run(pyr_cells, rd, 600 + i) for i, rd in enumerate(("a", "b"))}
print(f"GLM, pyramidal cells: {time.time() - t0:.0f} s")

In [ ]:
for rd in ("a", "b"):
    for mc in (1, 4):
        for reg, sig, first, tb in glm_summary(glm_pyr[rd], rd, mc):
            print(f"reading {rd}, sessions with at least {mc} cells, {reg}: earliest window from which all later are significant: {first} -> {tb} s before exit")
            if mc == 1:
                print(sig.round(4).to_string(index=False))
print("expected: CA1 0.1 +/- 0.1 s, PFC 0.3 +/- 0.1 s before exit")

**Result: blocked (methods underspecified).** The two alignments give different outcomes. With exit-aligned windows (reading a) choice is predicted from 0.2 s before exit in CA1 and 0.4 s in PFC, both within tolerance (0.1 and 0.3 +/- 0.1). With onset-aligned windows (reading b) both regions predict from 0.12 s before exit, within tolerance for CA1 only. Requiring at least 4 cells per session (a variant) gives 0.0 and 0.3 s under reading a, and 0.22 and 0.42 s under reading b.

## fig6e-glm-with-interneurons

> Adding interneurons made CA1 prediction accurate earlier, 0.2 s before exit, PFC prediction did
> not change, and resampling pyramidal cells to the same ensemble size gave 0.1 s again.

As fig6e-glm-choice-prediction with task-responsive pyramidal cells and interneurons (7 Hz). The
control resamples the session's pyramidal cells with replacement to the number of pyramidal cells
plus interneurons. To keep the run within its time limit the control uses 10 resamples per
session, each with 10 label shuffles (100 null fits, as for the other ensembles), where the claims
file reads 100 resamples as a standard choice.

In [ ]:
def all_cells(s, reg):
    d = U[(U.region == reg) & U.active_B & U.odor_active & (U.p_resp_a < 0.05) & (U["name"] == s.name)]
    return list(d.unit)


t0 = time.time()
glm_all = {rd: glm_run(all_cells, rd, 700 + i) for i, rd in enumerate(("a", "b"))}


def glm_resampled(reading, seed, n_res=10, n_shuf=10):
    rng = np.random.default_rng(seed)
    rows = []
    for s in S:
        for reg in ("CA1", "PFC"):
            pyr = pyr_cells(s, reg)
            n_all = len(all_cells(s, reg))
            if not pyr:
                continue
            acc = []
            for _ in range(n_res):
                pick = list(rng.choice(pyr, n_all, replace=True))
                acc.append(glm_session(s, pick, reading, rng, n_shuf))
            for k in range(len(acc[0])):
                vals = [a[k] for a in acc]
                rows.append(dict(name=s.name, region=reg, win=vals[0]["win"], cells=vals[0]["cells"], real=np.nanmean([v["real"] for v in vals]), shuf=np.nanmean([v["shuf"] for v in vals])))
    return pd.DataFrame(rows)


glm_res = {rd: glm_resampled(rd, 800 + i) for i, rd in enumerate(("a", "b"))}
print(f"GLM with interneurons and resampled control: {time.time() - t0:.0f} s")

In [ ]:
for label, res in (("pyramidal and interneurons", glm_all), ("resampled pyramidal control", glm_res)):
    for rd in ("a", "b"):
        for reg, sig, first, tb in glm_summary(res[rd], rd, 1):
            print(f"{label}, reading {rd}, {reg}: earliest window from which all later are significant: {first} -> {tb} s before exit")
            print(sig.round(4).to_string(index=False))
print("expected: CA1 with interneurons 0.2 +/- 0.1 s, PFC with interneurons 0.3 +/- 0.1 s, CA1 resampled control 0.1 +/- 0.1 s")

**Result: blocked (methods underspecified).** The claim depends on fig6e-glm-choice-prediction, which is blocked. For the record: with interneurons, reading a gives 0.4 s before exit in CA1 (expected 0.2 +/- 0.1) and 0.4 s in PFC (expected 0.3 +/- 0.1), and the resampled pyramidal control 0.1 s in CA1 (expected 0.1); reading b gives 0.12, 0.32 and 0.12 s.

## Spatial coding (fig7)

Sessions on the long T-maze: the Results name 26 sessions from 6 rats (the five full-maze rats and
CS39, whose stem is shorter but whose wells are at the ends of the T arms), the Methods the five
full-maze animals; both sets are shown, the Results' first. Cells are pyramidal cells (7 Hz, active)
that fire during the outbound runs (criterion, code lines 349 to 351). Outbound runs are
linearized as in `opa.outbound_runs`; rate maps are spikes per bin over occupancy per bin, smoothed
with a Gaussian of 2 bins; a trajectory has a field when the peak is at least 1 Hz, at least 2 s.d.
above the mean of a null of 500 circular shifts of the spikes within each run (the code's count),
and the field covers under 75% of the trajectory (criterion; Methods).

In [ ]:
SPATIAL = {"Results (6 rats, CS39 included)": FULL + SHORT_STEM, "Methods (5 full-maze rats)": FULL}
rng7 = np.random.default_rng(77)
RUNS = {s.name: opa.outbound_runs(s) for s in S if s.subject in FULL + SHORT_STEM}


def cell_maps(s, spk, runs, n_shuf=500, rng=rng7):
    """Per trajectory: occupancy, smoothed rate map, field test, and per-run counts by bin."""
    out = {}
    for side in ("L", "R"):
        rr = [r for r in runs if r["side"] == side]
        if not rr:
            continue
        occ = np.zeros(100)
        cnt_runs = []
        sh_idx = []
        for r in rr:
            occ += np.bincount(r["bins"], weights=r["dt"], minlength=100)
            k = np.searchsorted(r["t"], spk[(spk >= r["t"][0]) & (spk <= r["t"][-1] + r["dt"][-1])], side="right") - 1
            k = k[(k >= 0) & (k < len(r["t"]))]
            cnt_runs.append(np.bincount(r["bins"][k], minlength=100))
            sh_idx.append((k, len(r["t"]), r["bins"]))
        cnt = np.sum(cnt_runs, 0)
        okb = occ > 0
        rate = opa.gauss_smooth(np.where(okb, cnt / np.where(okb, occ, 1), 0.0))
        # null: circular shift of the spikes within each run
        null_peak = np.zeros(n_shuf)
        if cnt.sum() > 0:
            NC = np.zeros((n_shuf, 100))
            for k, n, bins in sh_idx:
                if not len(k):
                    continue
                sh = (k[None, :] + rng.integers(0, n, (n_shuf, 1))) % n
                b = bins[sh]
                NC += np.stack([np.bincount(row, minlength=100) for row in b])
            nr = opa.gauss_smooth(np.where(okb, NC / np.where(okb, occ, 1), 0.0), axis=1)
            null_peak = nr.max(1)
        peak = rate.max()
        width = opa.flood_width(rate) if peak > 0 else 0
        field = bool(peak >= 1.0 and peak >= null_peak.mean() + 2 * null_peak.std() and width < 75)
        out[side] = dict(occ=occ, rate=rate, peak=peak, width=width, field=field, runs=np.array(cnt_runs), run_occ=np.array([np.bincount(r["bins"], weights=r["dt"], minlength=100) for r in rr]), sparsity=opa.skaggs_sparsity(rate, occ), null_mean=null_peak.mean(), null_sd=null_peak.std())
    return out


t0 = time.time()
sp_rows = []
MAPS = {}
for s in S:
    if s.name not in RUNS:
        continue
    runs = RUNS[s.name]
    tr = s.trials
    for i, r in U[(U["name"] == s.name) & U.active_B & ~U.int7].iterrows():
        spk = s.units.spikes[r.unit]
        n_run = sum(((spk >= rr["t"][0]) & (spk <= rr["t"][-1])).sum() for rr in runs)
        if n_run == 0:
            continue
        mp = cell_maps(s, spk, runs)
        MAPS[i] = mp
        sp_rows.append(dict(idx=i, name=s.name, subject=s.subject, region=r.region, odor_active=bool(r.odor_active), field_L=mp.get("L", {}).get("field", False), field_R=mp.get("R", {}).get("field", False)))
SP = pd.DataFrame(sp_rows).set_index("idx")
SP["field"] = SP.field_L | SP.field_R
print(f"spatial maps: {len(SP)} cells in {SP.name.nunique()} sessions, {time.time() - t0:.0f} s")
print("runs per session:", {k: len(v) for k, v in RUNS.items()})

### fig7-spatial-field-counts

> Many CA1 and PFC cells had spatial fields on the track, 344 of 585 in CA1 and 159 of 288 in PFC.

In [ ]:
for label, subs in SPATIAL.items():
    d = SP[SP.subject.isin(subs)]
    print(label, "; sessions:", d.name.nunique())
    for reg, ek, en in (("CA1", 344, 585), ("PFC", 159, 288)):
        x = d[d.region == reg]
        print(f"  {reg}: {int(x.field.sum())} of {len(x)} cells with a field ({100 * x.field.mean():.1f}%); expected {ek} +/- {round(ek * 0.1)} of {en} +/- {round(en * 0.1)} ({100 * ek / en:.1f}%)")
    print("  per animal, fraction with a field:", d.groupby(["region", "subject"]).field.mean().round(2).to_dict())

**Result: reproduced.** On the Results' 26 sessions 365 of 591 CA1 and 163 of 278 PFC pyramidal cells have a field (expected 344 of 585 and 159 of 288, all four counts within tolerance); on the Methods' 19 sessions 339 of 545 and 147 of 250.

### fig7b-odor-active-more-fields

> Odor-period active cells were more likely to have spatial fields than inactive cells, in CA1 (71
> of 108 against 273 of 477) and PFC (83 of 128 against 76 of 160).

Binomial test of the active cells with fields against n active, with null probability the fraction
of inactive cells with fields, one-sided upper; P(X >= k) and the code's P(X > k) (criterion).

In [ ]:
for label, subs in SPATIAL.items():
    d = SP[SP.subject.isin(subs)]
    print(label)
    for reg in ("CA1", "PFC"):
        x = d[d.region == reg]
        a, ina = x[x.odor_active], x[~x.odor_active]
        k, n, p0 = int(a.field.sum()), len(a), ina.field.mean()
        print(f"  {reg}: active {k}/{n} ({100 * k / n:.0f}%) with fields, inactive {int(ina.field.sum())}/{len(ina)} ({100 * p0:.0f}%); P(X >= k) = {stats.binom.sf(k - 1, n, p0):.3g}, P(X > k) = {stats.binom.sf(k, n, p0):.3g}")
print("paper: CA1 71/108 against 273/477, p = 0.029; PFC 83/128 against 76/160, p = 2.7e-5")

**Result: partially reproduced.** Odor-active cells have fields more often than inactive cells in both regions and both session sets (CA1 66% against 61% and 70% against 61%; PFC 63% against 55% and 64% against 56%), but by the criterion's P(X >= k) only one region is significant in each set: PFC on the Results' sessions (p = 0.045; CA1 p = 0.15) and CA1 on the Methods' sessions (p = 0.042; PFC p = 0.059). With the code's P(X > k), both regions are significant on the Methods' sessions.

### fig7s1b-odor-active-higher-rate

> Odor-period active cells had higher mean firing rates during running than inactive cells, in CA1
> and PFC.

Mean rate of each cell over all task-epoch time at speed over 3 cm/s outside the sampling periods
(criterion); rank-sum of active against inactive cells.

In [ ]:
run_rate = {}
for s in S:
    if s.name not in RUNS:
        continue
    pos = s.pos
    dt = np.r_[np.diff(pos.t.values), 0]
    inodor = np.zeros(len(pos), bool)
    for a, b in zip(s.trials.start, s.trials.stop):
        inodor |= (pos.t.values >= a) & (pos.t.values < b)
    m = (pos.v.values > 3) & ~inodor & (dt < 0.1)
    tt, dd = pos.t.values[m], dt[m]
    for i in SP.index[SP["name"] == s.name]:
        spk = s.units.spikes[U.loc[i, "unit"]]
        run_rate[i] = opa.count_in(spk, tt, tt + dd).sum() / dd.sum()
SP["run_rate"] = pd.Series(run_rate)
for label, subs in SPATIAL.items():
    d = SP[SP.subject.isin(subs)]
    print(label)
    for reg in ("CA1", "PFC"):
        x = d[d.region == reg]
        a, ina = x[x.odor_active].run_rate, x[~x.odor_active].run_rate
        print(f"  {reg}: active median {a.median():.2f} Hz (n = {len(a)}), inactive {ina.median():.2f} Hz (n = {len(ina)}); rank-sum p = {stats.ranksums(a, ina).pvalue:.3g}")
print("paper: CA1 p = 2.8e-14 (n = 585), PFC p = 1.3e-5 (n = 288)")

**Result: reproduced.** Odor-active cells fire more during running in both regions (CA1 median 2.35 against 0.97 Hz, p = 1.5e-10; PFC 3.47 against 1.11 Hz, p = 1.4e-12; the Methods' sessions give p = 5.3e-10 and 1.6e-11).

### fig7c-field-properties-no-difference

> Place field width, peak rate and sparsity did not differ between odor-period active and inactive
> cells in either region.

Each outbound field counted separately; width by flood fill to 25% of the peak (in bins, each 1% of
the trajectory); peak on the smoothed map; Skaggs sparsity. Rank-sum of active against inactive
fields (criterion).

In [ ]:
fr_rows = []
for i, r in SP.iterrows():
    for side in ("L", "R"):
        mp = MAPS[i].get(side)
        if mp and mp["field"]:
            fr_rows.append(dict(idx=i, subject=r.subject, region=r.region, odor_active=r.odor_active, width=mp["width"], peak=mp["peak"], sparsity=mp["sparsity"]))
FR = pd.DataFrame(fr_rows)
for label, subs in SPATIAL.items():
    d = FR[FR.subject.isin(subs)]
    print(label)
    for reg in ("CA1", "PFC"):
        x = d[d.region == reg]
        for prop in ("width", "peak", "sparsity"):
            a, ina = x[x.odor_active][prop], x[~x.odor_active][prop]
            print(f"  {reg} {prop}: active median {a.median():.3f} (n = {len(a)}), inactive {ina.median():.3f} (n = {len(ina)}); rank-sum p = {stats.ranksums(a, ina).pvalue:.3g}")
print("paper: fields CA1 n = 452, PFC n = 207; width p = 0.15, 0.92; peak p = 0.14, 0.08; sparsity p = 0.70, 0.88")

**Result: not reproduced.** Four of the six tests find a difference: CA1 fields of odor-active cells are narrower (median 14 against 21 bins, p = 1.6e-7) and sparser (p = 2.8e-5), and PFC fields of odor-active cells have higher peaks (11.9 against 7.4 Hz, p = 1.2e-4) and sparsity (p = 3.2e-10); CA1 peak (p = 0.82) and PFC width (p = 0.96) do not differ. The Methods' sessions give the same pattern.

### fig7s1e-trajectory-selective-odor-active

> Trajectory-selective cells were more likely than non-selective cells to be active in the odor
> period.

Trajectory selective (Methods): the correlation of the left and right outbound tuning curves below
the 5th percentile of a null from 500 shuffles of trajectory identity across runs. Cells with
runs on both trajectories. Binomial test: the selective cells that are odor active against the
share odor active among non-selective cells, one-sided upper (criterion).

In [ ]:
def traj_selective(mp, n_shuf=500, rng=rng7):
    if "L" not in mp or "R" not in mp:
        return np.nan, np.nan
    C = np.vstack([mp["L"]["runs"], mp["R"]["runs"]]).astype(float)
    O = np.vstack([mp["L"]["run_occ"], mp["R"]["run_occ"]])
    lab = np.r_[np.ones(len(mp["L"]["runs"]), bool), np.zeros(len(mp["R"]["runs"]), bool)]

    def corr(lb):
        a = gauss_smooth_rate(C[lb].sum(0), O[lb].sum(0))
        b = gauss_smooth_rate(C[~lb].sum(0), O[~lb].sum(0))
        if a.std() == 0 or b.std() == 0:
            return np.nan
        return np.corrcoef(a, b)[0, 1]

    obs = corr(lab)
    null = np.array([corr(rng.permutation(lab)) for _ in range(n_shuf)])
    null = null[np.isfinite(null)]
    return obs, (null < obs).mean() if len(null) and np.isfinite(obs) else np.nan


def gauss_smooth_rate(c, o):
    ok = o > 0
    return opa.gauss_smooth(np.where(ok, c / np.where(ok, o, 1), 0.0))


ts = {i: traj_selective(MAPS[i]) for i in SP.index}
SP["traj_r"] = [ts[i][0] for i in SP.index]
SP["traj_q"] = [ts[i][1] for i in SP.index]
SP["traj_sel"] = SP.traj_q < 0.05
for label, subs in SPATIAL.items():
    d = SP[SP.subject.isin(subs) & SP.traj_q.notna()]
    print(label)
    for reg in ("CA1", "PFC"):
        x = d[d.region == reg]
        sel, non = x[x.traj_sel], x[~x.traj_sel]
        k, n, p0 = int(sel.odor_active.sum()), len(sel), non.odor_active.mean()
        print(f"  {reg}: trajectory selective {n} of {len(x)}; odor active among selective {k}/{n} ({100 * k / max(n, 1):.0f}%), among non-selective {100 * p0:.0f}%; binomial P(X >= k) = {stats.binom.sf(k - 1, n, p0):.3g}")
print("paper: CA1 p = 3.0e-4, PFC p = 0.02")

**Result: not reproduced.** By the Methods' rule 369 of 499 CA1 and 165 of 268 PFC cells are trajectory selective. Selective CA1 cells are less often odor active than non-selective ones (17% against 25%, p = 1.0); in PFC the difference is in the paper's direction but not significant on the Results' sessions (45% against 40%, p = 0.11) and is on the Methods' sessions (p = 0.049).

### fig7d-trajectory-choice-unrelated

> Among choice-selective cells, the preferred run trajectory was unrelated to the preferred choice
> in the odor period.

Trajectory selectivity index (Methods): the mean rate on each outbound run from 0.5 s after odor
port exit once speed exceeds 3 cm/s to the run's end (goal reached, or speed under 3 cm/s for more
than 0.5 s), averaged per trajectory, (left - right)/(left + right). Rank-sum of the index between
choice-selective cells preferring the left choice (SI > 0) and those preferring the right.

In [ ]:
def traj_si(s, spk):
    pos = s.pos
    t, v = pos.t.values, pos.v.values
    rates = {"L": [], "R": []}
    for r in RUNS[s.name]:
        m = (t >= r["t0"] + 0.5) & (t <= r["t1"])
        if not m.any():
            continue
        tt, vv = t[m], v[m]
        k0 = np.nonzero(vv > 3)[0]
        if not len(k0):
            continue
        a = tt[k0[0]]
        slow = vv < 3
        end = r["t1"]
        for k in range(k0[0], len(tt)):
            if slow[k]:
                j = k
                while j < len(tt) and slow[j]:
                    j += 1
                if tt[min(j, len(tt) - 1)] - tt[k] > 0.5:
                    end = tt[k]
                    break
        if end - a <= 0:
            continue
        rates[r["side"]].append(opa.count_in(spk, a, end) / (end - a))
    L, R = np.mean(rates["L"]) if rates["L"] else np.nan, np.mean(rates["R"]) if rates["R"] else np.nan
    return (L - R) / (L + R) if (L + R) > 0 else np.nan


for label, subs in SPATIAL.items():
    print(label)
    for reg in ("CA1", "PFC"):
        d = cellset(reg, "pyr")
        d = d[d.subject.isin(subs) & d.selective.astype(bool)]
        tsi = np.array([traj_si(sess[r["name"]], sess[r["name"]].units.spikes[r.unit]) for _, r in d.iterrows()])
        prefL = d.si_c.values > 0
        a, b = tsi[prefL], tsi[~prefL]
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        print(f"  {reg}: choice-selective cells {len(d)}; trajectory SI median {np.median(a):.3f} (prefer left choice, n = {len(a)}) against {np.median(b):.3f} (right, n = {len(b)}); rank-sum p = {stats.ranksums(a, b).pvalue:.3g}")
print("paper: CA1 p = 0.26, PFC p = 0.20")

**Result: not reproduced.** In CA1 the trajectory index does not differ between cells preferring the two choices (p = 0.24 and 0.12), but in PFC it does (p = 0.010 on the Results' sessions, 0.0061 on the Methods'), with cells preferring the left choice also preferring the left trajectory (median 0.05 against -0.13).

### fig7e-odor-decoder-first-quintile and fig7f-spatial-decoder-all-quintiles

Memoryless Poisson Bayesian decoder (Methods, Bayesian decoder): P(choice | spikes) proportional to
the product over cells of f_i(choice)^n_i times exp(-tau sum f_i(choice)), with a flat prior; the
decoded choice the more probable one. Runs from 0.5 s after odor port exit at speed over 3 cm/s,
linearized as above and cut into five equal fifths of the trajectory. Accuracy per session and
quintile is the fraction of runs (with time in that quintile) decoded as the chosen side. A rate of
zero is floored at 0.01 Hz (a standard choice). Null: 200 shuffles of the training labels; rank-sum
over sessions of the real accuracy against the mean shuffled accuracy (legend; the code's paired
signed-rank as a variant).

In [ ]:
RUNS5 = {s.name: opa.outbound_runs(s, start_delay=0.5) for s in S if s.subject in FULL + SHORT_STEM}


def run_quintile_counts(s, units, runs):
    """counts (runs x cells x 5) and durations (runs x 5) of each run in each fifth."""
    C = np.zeros((len(runs), len(units), 5))
    T = np.zeros((len(runs), 5))
    for k, r in enumerate(runs):
        q = np.minimum(r["bins"] // 20, 4)
        T[k] = np.bincount(q, weights=r["dt"], minlength=5)
        for j, u in enumerate(units):
            spk = s.units.spikes[u]
            kk = np.searchsorted(r["t"], spk[(spk >= r["t"][0]) & (spk <= r["t"][-1] + r["dt"][-1])], side="right") - 1
            kk = kk[(kk >= 0) & (kk < len(r["t"]))]
            C[k, j] = np.bincount(q[kk], minlength=5)
    return C, T


def decode(f, C, T):
    """f: (2 choices, cells) rates; C: (runs, cells) counts; T: (runs,) durations. Returns 0/1 choice."""
    lf = np.log(np.maximum(f, 0.01))
    ll = C @ lf.T - T[:, None] * np.maximum(f, 0.01).sum(1)[None, :]
    return np.argmax(ll, 1)


def p_rank(real, shuf):
    return stats.ranksums(real, shuf).pvalue, stats.wilcoxon(real, shuf).pvalue if (np.asarray(real) != np.asarray(shuf)).any() else np.nan


rng8 = np.random.default_rng(88)
dec_e = []
for s in S:
    if s.name not in RUNS5:
        continue
    runs = RUNS5[s.name]
    tr = s.trials
    for reg in ("CA1", "PFC"):
        cells = U[(U["name"] == s.name) & (U.region == reg) & U.active_B & U.odor_active & (U.p_resp_a < 0.05)]
        if not len(cells) or len(runs) < 10:
            continue
        units = list(cells.unit)
        ok = tr.side.values != ""
        y = (tr.side.values[ok] == "R").astype(int)
        R = np.stack([opa.count_in(s.units.spikes[u], tr.start.values[ok], tr.stop.values[ok]) / tr.dur.values[ok] for u in units], 1)
        C, T = run_quintile_counts(s, units, runs)
        ytest = np.array([r["side"] == "R" for r in runs]).astype(int)

        def acc_for(lab):
            f = np.stack([R[lab == 0].mean(0), R[lab == 1].mean(0)])
            out = []
            for q in range(5):
                m = T[:, q] > 0
                out.append(np.mean(decode(f, C[m, :, q], T[m, q]) == ytest[m]) if m.any() else np.nan)
            return np.array(out)

        real = acc_for(y)
        shuf = np.nanmean([acc_for(rng8.permutation(y)) for _ in range(200)], 0)
        for q in range(5):
            dec_e.append(dict(name=s.name, subject=s.subject, region=reg, q=q + 1, cells=len(units), real=real[q], shuf=shuf[q]))
DE = pd.DataFrame(dec_e)

#### fig7e-odor-decoder-first-quintile

> A decoder trained on odor-period activity decoded the choice from run activity in the first fifth
> of the track, nearest the odor port, but at chance thereafter.

Trained on the spikes of the sampling period of every trial with a known choice, tested on the
runs; task-responsive cells (legend), pyramidal cells and interneurons.

In [ ]:
for label, subs in SPATIAL.items():
    print(label)
    for reg in ("CA1", "PFC"):
        for q in range(1, 6):
            d = DE[(DE.region == reg) & (DE.q == q) & DE.subject.isin(subs)].dropna()
            p1, p2 = p_rank(d.real, d.shuf)
            print(f"  {reg} quintile {q}: sessions {len(d)}, accuracy {d.real.mean():.3f} against shuffled {d.shuf.mean():.3f}; rank-sum p = {p1:.3g}; signed-rank p = {p2:.3g}")

**Result: not reproduced.** The decoder trained on the sampling period does not decode the choice from CA1 run activity in any quintile (quintile 1: 0.53 against 0.50 shuffled, p = 0.53). In PFC it does so in quintile 1 on the Methods' sessions only (0.58 against 0.51, p = 0.031; on the Results' sessions p = 0.19), and in no later quintile.

#### fig7f-spatial-decoder-all-quintiles

> Decoders trained on the spatially active ensemble in each fifth of the track decoded the upcoming
> choice at every position along the trajectory.

Trained and tested on run activity in each fifth, five-fold cross-validation over runs, 200 label
shuffles (criterion). Readings for "spatially active": (a) pyramidal cells with a place field on
either outbound trajectory (fig7-spatial-field-counts); (b) cells firing in run on at least 10 runs
(code), pyramidal cells and interneurons.

In [ ]:
def cv_decode(C, T, y, rng, k=5):
    n = len(y)
    order = rng.permutation(n)
    folds = np.array_split(order, k)
    correct_ = np.zeros(n)
    used = np.zeros(n, bool)
    for f in folds:
        trn = np.setdiff1d(order, f)
        if len(np.unique(y[trn])) < 2:
            continue
        rates = np.stack([C[trn][y[trn] == c].sum(0) / max(T[trn][y[trn] == c].sum(), 1e-9) for c in (0, 1)])
        m = T[f] > 0
        if m.any():
            correct_[f[m]] = decode(rates, C[f[m]], T[f[m]]) == y[f[m]]
            used[f[m]] = True
    return correct_[used].mean() if used.any() else np.nan


dec_f = []
for s in S:
    if s.name not in RUNS5:
        continue
    runs = RUNS5[s.name]
    if len(runs) < 10:
        continue
    y = np.array([r["side"] == "R" for r in runs]).astype(int)
    for reg in ("CA1", "PFC"):
        cand = U[(U["name"] == s.name) & (U.region == reg) & U.active_B]
        allu = list(cand.unit)
        if not allu:
            continue
        C, T = run_quintile_counts(s, allu, runs)
        nruns_active = (C.sum(2) > 0).sum(0)
        sets = {
            "a": [j for j, i in enumerate(cand.index) if i in SP.index and SP.loc[i, "field"]],
            "b": [j for j in range(len(allu)) if nruns_active[j] >= 10],
        }
        for rd, js in sets.items():
            if not js:
                continue
            for q in range(5):
                Cq, Tq = C[:, js, q], T[:, q]
                real = cv_decode(Cq, Tq, y, rng8)
                shuf = np.nanmean([cv_decode(Cq, Tq, rng8.permutation(y), rng8) for _ in range(200)])
                dec_f.append(dict(name=s.name, subject=s.subject, region=reg, reading=rd, q=q + 1, cells=len(js), real=real, shuf=shuf))
DF = pd.DataFrame(dec_f)

In [ ]:
for label, subs in SPATIAL.items():
    print(label)
    for rd in ("a", "b"):
        for reg in ("CA1", "PFC"):
            for q in range(1, 6):
                d = DF[(DF.region == reg) & (DF.q == q) & (DF.reading == rd) & DF.subject.isin(subs)].dropna()
                p1, p2 = p_rank(d.real, d.shuf)
                print(f"  reading {rd}, {reg} quintile {q}: sessions {len(d)}, accuracy {d.real.mean():.3f} against shuffled {d.shuf.mean():.3f}; rank-sum p = {p1:.3g}; signed-rank p = {p2:.3g}")

**Result: reproduced.** Decoders trained on run activity decode the choice in all five quintiles in both regions, under both cell readings and both session sets (accuracy 0.75 to 0.99 against about 0.50 shuffled; every rank-sum p below 1e-4).

### fig7s1f-ensemble-rate-near-port

> In CA1 the firing rate of choice-selective ensembles during outbound runs was highest within 20 cm
> of the odor port and fell thereafter.

Mean over cells of each cell's linearized outbound rate (the mean of its left and right maps), the
cells being (Results) the choice-selective task-responsive pyramidal cells or (legend) the
odor-period active pyramidal cells, of the spatial sessions (Results' set). Null from circularly
shifting each cell's curve by a random number of bins, 1000 times; a position is marked when the
ensemble rate exceeds the null's 99th percentile (criterion). Bin k covers k% of a route whose
median length is printed; 20 cm is that fraction of the route.

In [ ]:
route = np.median([r["length"] for v in RUNS.values() for r in v])
cut = int(np.ceil(100 * 20 / route))
print(f"median route length {route:.0f} cm; the first 20 cm are bins 0 to {cut - 1}")
for label, sel in (("choice selective (Results)", lambda d: d.selective.astype(bool) & d.odor_active & (d.p_resp_a < 0.05)), ("odor period active (legend)", lambda d: d.odor_active)):
    for reg in ("CA1", "PFC"):
        idx = [i for i in SP.index if SP.loc[i, "region"] == reg and sel(U.loc[[i]]).iloc[0]]
        curves = np.array([np.nanmean([MAPS[i][sd]["rate"] for sd in ("L", "R") if sd in MAPS[i]], 0) for i in idx])
        if not len(curves):
            continue
        ens = curves.mean(0)
        null = np.array([np.mean([np.roll(c, rng8.integers(100)) for c in curves], 0) for _ in range(1000)])
        thr = np.percentile(null, 99, axis=0)
        marked = np.nonzero(ens > thr)[0]
        print(f"{label}, {reg}: cells {len(curves)}; peak at bin {int(np.argmax(ens))} ({np.argmax(ens) * route / 100:.0f} cm); mean rate within 20 cm {ens[:cut].mean():.2f} Hz, beyond {ens[cut:].mean():.2f} Hz; marked bins: {marked.tolist()}")

**Result: reproduced.** The CA1 choice-selective ensemble (25 cells) peaks at the start of the route (bin 0) and its rate within the first 20 cm (4.0 Hz) exceeds that beyond (1.2 Hz); bins 0 to 6 exceed the 99th percentile of the null. The odor-active set gives the same (101 cells, 3.5 against 2.0 Hz, bins 0 to 9).

## Summary

| Claim | Finding | Outcome |
|---|---|---|
| fig1b-performance | Performance 77.0% over rats | reproduced |
| fig1c-sampling-duration | Sampling 0.82 s, same on correct and incorrect trials | reproduced |
| fig1d-turn-direction | Turn at the port matches the chosen arm | partially reproduced |
| fig1s1e-sniff-rate | Sniff rate rises during sampling | blocked: data missing |
| fig1s1f-air-performance | Performance lower on air-cued sessions | blocked: data missing |
| fig2d-beta-power-increase | Beta power rises during sampling | reproduced |
| fig2d-rr-power-no-increase | RR power does not rise; CA1 falls | not reproduced |
| fig2e-beta-coherence-increase | Beta coherence rises, all pairs | reproduced |
| fig2e-rr-coherence-increase | RR coherence rises, all pairs | partially reproduced |
| fig2e-beta-coherence-odor-vs-reward | Beta coherence higher at the port than at reward | reproduced |
| fig2e-rr-coherence-odor-vs-reward | RR coherence higher at the port for OB pairs only | blocked: methods underspecified |
| fig2d-power-odor-vs-reward | Beta power differs odor vs reward; RR only in OB | partially reproduced |
| fig2de-correct-incorrect-no-difference | No power or coherence difference, correct vs incorrect | blocked: methods underspecified |
| fig2g-air-beta-coherence | Beta coherence lower on air trials | blocked: data missing |
| fig2f-swr-odor-vs-reward | Ripples rare at the port, frequent at reward | reproduced |
| fig2s1f-rr-beta-coupling | RR-beta phase-amplitude coupling | reproduced |
| fig2s2d-phase-offsets | CA1 leads PFC (beta, RR) and OB (RR) | partially reproduced |
| fig2s1e-run-theta-shift | Peak frequency shifts to theta during run | blocked: methods underspecified |
| unit-counts | Unit, cell-type and selection counts | partially reproduced |
| fig3-choice-selective-fraction | About a third of cells choice selective | partially reproduced |
| fig3ij-si-anticorrelated | Correct and incorrect SI anti-correlated | partially reproduced |
| fig3s1e-nonselective-si | Non-selective cells (Results against legend) | blocked: methods underspecified |
| fig3s1c-odor-vs-reward-rate | Rates differ, odor vs reward | reproduced |
| fig4c-interneuron-si | CA1 interneuron SI anti-correlated | not reproduced |
| fig4d-ccg-lags | PFC interneurons lead in CA1-PFC CCGs | not reproduced |
| fig5c-local-locking | Local RR and beta locking | partially reproduced |
| fig5d-cross-region-beta | CA1 interneurons lock to PFC beta | reproduced |
| fig5gh-phase-preference | Consistent preferred phases | partially reproduced |
| fig5-rr-more-than-beta | More and deeper RR locking than beta | partially reproduced |
| fig5s1de-beta-rr-overlap | Beta and RR locking overlap at chance | reproduced |
| fig5s1fh-selective-and-locked | Selective and locked at chance | reproduced |
| fig5e-beta-mvl-correct-incorrect | Beta locking weaker on incorrect trials | reproduced |
| fig5f-rr-mvl-no-difference | RR locking same on correct and incorrect | reproduced |
| fig5s2-ca1-int-pfc-beta-accuracy | CA1 interneuron to PFC beta locking and accuracy | reproduced |
| fig6d-discrimination-time | Ensemble discrimination times | not reproduced |
| fig6c-even-odd-control | Even-odd control never significant | not reproduced |
| fig6e-glm-choice-prediction | GLM predicts choice before exit | blocked: methods underspecified |
| fig6e-glm-with-interneurons | Interneurons make CA1 prediction earlier | blocked: methods underspecified |
| fig7-spatial-field-counts | Cells with place fields | reproduced |
| fig7b-odor-active-more-fields | Odor-active cells more often have fields | partially reproduced |
| fig7s1b-odor-active-higher-rate | Odor-active cells fire more in run | reproduced |
| fig7c-field-properties-no-difference | Field properties do not differ | not reproduced |
| fig7s1e-trajectory-selective-odor-active | Trajectory-selective cells more often odor active | not reproduced |
| fig7d-trajectory-choice-unrelated | Trajectory and choice preference unrelated | not reproduced |
| fig7e-odor-decoder-first-quintile | Odor-trained decoder works in the first fifth only | not reproduced |
| fig7f-spatial-decoder-all-quintiles | Run-trained decoder works everywhere | reproduced |
| fig7s1f-ensemble-rate-near-port | CA1 selective ensemble rate highest near the port | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.**
- The trial table holds no odor identity and no chosen well. The chosen side is read from the
  position (the well approached after exit; it agrees with the rewarded well on 99.85% of rewarded
  trials), and the odor is the chosen side on rewarded trials and the other side otherwise. Which
  side is odor 1 is not known.
- The files hold no spike waveforms, so putative interneurons are defined by rate alone (over
  7 Hz; the code's 8.5 and 9.5 Hz as variants). This over-counts PFC interneurons and affects
  every claim that separates cell types.
- The units' regions are read from the tetrode number in the units table, not from the electrode
  row it should index (data check).
- The LFP is read in windows around each trial, around each reward visit and at 60 random times per
  day, not whole; the epoch statistics for z-scoring and the ripple threshold come from those 60
  windows (120 s per day), from task epochs, since the files hold no sleep.
- Filters are Butterworth band-passes applied forward and backward (order 2 for beta and RR, 4 for
  ripples); the paper gives "zero-phase IIR" without an order.
- The paper's two code pipelines are run side by side where the claims file names them; where they
  disagree in outcome the claim is blocked, as the claims file says.
- The GLM uses the code's logit link, not the Methods' log link, and the GLM control uses 10
  resamples per session.
- The code's exit times from DIO events, its 200 jitters for the cross-correlograms and the
  thermocouple analysis are not available or not run.

**What was found about the dataset.**
- The units table's `electrodes` column holds tetrode numbers, not rows of the electrode table.
  Read as rows, 131 units fall on OB tetrodes, which carried no units, and regions come out as 540
  CA1 and 752 PFC; read as tetrode numbers, 915 and 508, against the paper's 934 and 508 active
  units.
- The dandiset holds the active units only (915 and 508), not the 1,309 and 717 the paper reports,
  without saying so.
- The trial table has no odor identity and no chosen well; unrewarded trials have no reward times.
- There are no spike waveforms or widths, no thermocouple channel, no air-only sessions and no
  sleep epochs.
- Electrode group names and labels disagree (group Tetrode5 is labelled tetrode6, and so on) in all
  38 files.
- The acquisition ElectricalSeries is a copy of the LFP in every file, which doubles the dandiset's
  size (95 GB); the LFP is stored uncompressed and unchunked as float64.
- CS39 session 02 has positions on a different scale from the other CS39 days (range 59 by 25
  against about 105 by 45 cm), so its stated unit of centimeters is doubtful.
- A few CS42 trials end after the last LFP sample.